# Automação — Energia Injetada (Verginia Participações: Chácara Nelson + Nelson Souza)

Notebook dedicado aos demonstrativos Copel das UCs da **Verginia Participações**, que têm um
layout diferente do demonstrativo PR (lojas):

| | Demonstrativo PR (lojas) | Chácara Nelson / Nelson Souza |
|---|---|---|
| Sub-colunas por métrica | 2 (`PT` \| `FP`) — 17 colunas | 1 (`TP`) — 10 colunas |
| Planilha de partida | `Acompanhamento Energia Injetada PR.xlsx` | **nenhuma**Relatório único** (v3): os dois demonstrativos (Chácara Nelson e Nelson Souza) são lidos **juntos** e geram
uma planilha, um painel **no padrão do painel PR** e um relatório PDF em `saida/`. O saldo **não é somado**
entre origens: cada seção do demonstrativo (UC × geradora de origem) é uma linha — a Chácara aparece como
`(GERADORA)` (geração própria) e como beneficiária da Nelson Souza:

- `Acompanhamento Energia Injetada Verginia Participacoes.xlsx`
- `painel_energia_injetada_verginia_participacoes.png/.pdf`
- `relatorio_energia_injetada_verginia_participacoes.pdf` (rateio de cada geradora)

**Identificação das UCs:** os demonstrativos novos não trazem mais o *UC Conces.*; a chave passa a ser o
número da **UC** (15 dígitos do PDF). O nome de cada UC vem da aba `Unidades Consumidoras` da planilha
`Acompanhamento Energia Injetada Verginia Participacoes.xlsx` (pasta do notebook): coluna `uc` (com ou sem
pontos, ex.: `000.635.494.503.104`) e coluna `nome`. UC fora da aba aparece como `UC <nº> - <cidade>/<bairro>`.

ada Nelson Souza.xlsx`

### Estrutura dos PDFs

Cada página é uma **seção** = (UC, geradora de origem dos créditos). A mesma UC pode aparecer em
mais de uma seção:

- **Chácara Nelson** (UC Conces. 110761898): seção *Geradora* (geração própria, 100%) + seção
  *Beneficiária* dos créditos da geradora Nelson Souza (32%).
- **Nelson Souza** (UC Conces. 6205976): seção *Geradora* + as beneficiárias dela — UC 53894430
  (Curitiba, 68%) e a própria Chácara (32%).

A seção repetida nos dois PDFs (Chácara beneficiária da Nelson Souza) é lida uma vez só.
Por isso a aba-mestre guarda **1 linha por UC × geradora de origem × mês**, e o saldo de uma UC é a
**soma dos saldos de todas as origens**. A *Energia Ativa* (consumo) se repete em todas as seções
da UC; a coluna `consumo` a registra **uma única vez** por UC/mês para não somar em dobro.

### Fluxo

1. **Extração** — lê cada página por *rótulo* de coluna (funciona com 1 ou 2 sub-colunas).
2. **Validação** — confere `saldo_final = saldo_anterior + cred_receb + injetada − crédito utilizado − transferido`
   e a continuidade saldo final → saldo anterior do mês seguinte.
3. **Planilha** — se já existir em `saida/`, o histórico dela é **mesclado** com o PDF (meses antigos
   que saíram da janela de 13 meses do PDF são mantidos; meses repetidos assumem o valor do PDF).
   O arquivo é então regravado com as abas: resumos, uma aba por UC, `Demonstrativo` e
   `Créditos a Expirar`.
4. **Rateio anual** (v2) — % de cada beneficiária para o próximo ciclo de 12 meses (mesmo bloco dos notebooks Copel/CELESC).
5. **Relatório PDF por demonstrativo** (v2) — painel + rateio + gráficos + métodos + premissas, e abas `Rateio *` em cada planilha.

## 1. Configuração

In [ ]:
from __future__ import annotations
import io
import logging
import re
import sys
import unicodedata
import datetime as dt
from pathlib import Path

import pandas as pd
import pdfplumber
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

# ---------------------------------------------------------------- CAMINHOS
PASTA_PDF = Path("demonstativos")
OUT_DIR   = Path("saida"); OUT_DIR.mkdir(exist_ok=True)

# Relatório ÚNICO: todos os demonstrativos abaixo são lidos juntos -> 1 planilha, 1 painel, 1 relatório.
NOME_RELATORIO = "Verginia Participacoes"
DEMONSTRATIVOS = [
    PASTA_PDF / "demonstrativo-chacara-nelson.pdf",
    PASTA_PDF / "demonstrativo-nelson-souza.pdf",
]

# Planilha com a aba "Unidades Consumidoras" (UC -> nome exibido). Colunas: uc | nome.
CADASTRO_XLSX = Path(f"Acompanhamento Energia Injetada {NOME_RELATORIO}.xlsx")
ABA_CADASTRO  = "Unidades Consumidoras"

def xlsx_saida(nome: str = NOME_RELATORIO) -> Path:
    return OUT_DIR / f"Acompanhamento Energia Injetada {nome}.xlsx"

# ---------------------------------------------------------------- PAINEL (Etapa 5)
EMPRESA = "Verginia Participacoes e Administracao Ltda"
UF      = "Paraná"

def relatorio_saida(nome: str) -> Path:         # (v2) relatório completo do demonstrativo
    return OUT_DIR / f"relatorio_energia_injetada_{normalizar(nome).replace(' ', '_')}.pdf"

def painel_saida(nome: str) -> Path:            # sem extensão (gera .png e .pdf)
    return OUT_DIR / f"painel_energia_injetada_{normalizar(nome).replace(' ', '_')}"

# Status do saldo no painel (padrão PR): pelo SALDO em kWh
LIM_SUPERAVIT   = 2000        # saldo > X kWh                       -> "Superávit"
LIM_CONFORTAVEL = 800         # LIM_CONFORTAVEL < saldo <= LIM_SUPERAVIT -> "Confortável"
TAG_GERADORA    = "(GERADORA)"  # sufixo das linhas de seção Geradora (como "80 - INDUSTRIA (GERADORA)")

# Status do saldo por AUTONOMIA (painel antigo, não usado no padrão PR) = saldo ÷ consumo médio mensal (meses de consumo cobertos)
LIM_MESES_SUPERAVIT   = 12    # autonomia > 12 meses           -> "Superávit"
LIM_MESES_CONFORTAVEL = 3     # 3 < autonomia <= 12            -> "Confortável"
LIM_BAIXO             = 1     # saldo >= 1 kWh e autonomia <= 3 -> "Saldo baixo"; abaixo -> "Crítico / Zero"

# % ideal: com ESTRATEGIA_RATEIO = "W" (célula 4.0) vem do rateio anual; abaixo, parâmetros legados C/A
HORIZONTE_MESES    = 12       # meses até a próxima revisão do rateio: 12 (1x/ano) ou 6 (2x/ano)
CONSIDERAR_SALDO   = True     # True = estratégia C (desconta o saldo); False = estratégia A (só consumo)
JANELA_MEDIA_MESES = 12       # meses mais recentes na média de consumo (None = todo o histórico)
MESES_BUFFER       = 0.5      # margem de segurança da estratégia C
PISO_PCT           = 0.0      # % mínima por beneficiária (0 = sem piso)
VALIDADE_CREDITOS_MESES = 60  # validade dos créditos de energia (aviso de risco de expiração)

# ---------------------------------------------------------------- LOGGING
logger = logging.getLogger("energia_nelson")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_h = logging.StreamHandler(sys.stdout)
_h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
logger.addHandler(_h)


# ---------------------------------------------------------------- NOMES DAS UCs
def uc_int(v) -> int | None:
    """'000.635.494.503.104' / '000635494503104' / 635494503104 -> 635494503104."""
    dig = re.sub(r"\D", "", str(v if v is not None else ""))
    return int(dig) if dig else None


def ler_cadastro_ucs(xlsx: Path = CADASTRO_XLSX, aba: str = ABA_CADASTRO) -> dict:
    """{UC (int) -> nome} da aba `Unidades Consumidoras`. Cabeçalho: coluna da UC (`uc`, `uc_conces`,
    `Unidade Consumidora`…) e coluna do nome (`nome`, `nome_loja`…); sem cabeçalho reconhecível = A e B."""
    if not xlsx.exists():
        logger.warning("Cadastro %s não encontrado — UCs aparecerão pelo número.", xlsx.name)
        return {}
    wb = openpyxl.load_workbook(io.BytesIO(xlsx.read_bytes()), read_only=True, data_only=True)
    if aba not in wb.sheetnames:
        logger.warning("%s não tem a aba %r — UCs aparecerão pelo número.", xlsx.name, aba)
        return {}
    linhas = [r for r in wb[aba].iter_rows(values_only=True) if any(c is not None for c in r)]
    if not linhas:
        return {}
    hdr = [re.sub(r"\s+", " ", unicodedata.normalize("NFKD", str(c or "")).encode("ascii", "ignore")
                  .decode()).strip().lower() for c in linhas[0]]
    col_uc = next((j for j, c in enumerate(hdr)
                   if c.startswith("uc") or c in ("unidade consumidora", "unidade_consumidora")), None)
    col_nome = next((j for j, c in enumerate(hdr)
                     if j != col_uc and any(k in c for k in ("nome", "loja", "unidade", "descri"))), None)
    if col_uc is None or col_nome is None:          # sem cabeçalho: A = UC, B = nome
        col_uc, col_nome, dados = 0, 1, linhas
    else:
        dados = linhas[1:]
    mapa = {}
    for r in dados:
        uc = uc_int(r[col_uc] if col_uc < len(r) else None)
        nome = r[col_nome] if col_nome < len(r) else None
        if uc and nome not in (None, ""):
            mapa[uc] = str(nome).strip()
    logger.info("Cadastro %s / %s: %d UC(s).", xlsx.name, aba, len(mapa))
    return mapa


# UC (int) -> nome exibido. UCs fora do cadastro recebem "UC <nº> - <cidade>/<bairro>".
NOMES_UC = ler_cadastro_ucs()
logger.info("Configuração carregada: relatório %r com %d demonstrativo(s).", NOME_RELATORIO, len(DEMONSTRATIVOS))

## 2. Extração do PDF

### 2.1 Utilidades de parsing

In [ ]:
def limpar_numero(txt) -> float:
    """Texto -> float tolerando formato BR. '.' só é milhar quando há ',' na string."""
    if txt is None:
        return 0.0
    if isinstance(txt, (int, float)):
        return float(txt)
    s = str(txt).strip().replace("kWh", "").replace("%", "").strip()
    if s in ("", "-", "—"):
        return 0.0
    if "," in s:
        s = s.replace(".", "").replace(",", ".")
    s = re.sub(r"[^0-9.\-]", "", s)
    try:
        return float(s) if s not in ("", "-", ".", "-.") else 0.0
    except ValueError:
        return 0.0


def parse_referencia(txt) -> dt.datetime | None:
    """'08/2026' -> datetime(2026, 8, 1)."""
    m = re.fullmatch(r"\s*(\d{1,2})\s*/\s*(\d{4})\s*", str(txt or ""))
    if m and 1 <= int(m.group(1)) <= 12:
        return dt.datetime(int(m.group(2)), int(m.group(1)), 1)
    return None


def parse_data(txt) -> dt.datetime | None:
    """'22/09/2026' -> datetime(2026, 9, 22)."""
    try:
        return dt.datetime.strptime(txt, "%d/%m/%Y")
    except (TypeError, ValueError):
        return None


def normalizar(s) -> str:
    """Minúsculo, sem acentos e com espaços simples — para casar rótulos."""
    s = unicodedata.normalize("NFKD", str(s or "")).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip().lower()

### 2.2 Cabeçalho e rodapé da página

Do cabeçalho: **UC** (nº de 15 dígitos — é a chave da UC; os demonstrativos novos não têm mais *UC Conces.*), cidade/bairro, **tipo da seção** (Geradora/Beneficiária) e a
**geradora de origem** dos créditos. Do rodapé: créditos expirados, próxima parcela a expirar e data
de emissão.

In [ ]:
def ler_cabecalho(texto: str) -> dict:
    info = dict(uc=None, uc_conces=None, cidade=None, bairro=None, tipo=None,
                geradora_uc_conces=None, expirados_kwh=0.0, prox_expirar_kwh=0.0,
                prox_expirar_data=None, emitido_em=None)
    # "UC. : 000635494503104" — a UC é a chave (campo `uc_conces` guarda a UC como número)
    m = re.search(r"^UC\.?\s*:\s*(\d+)", texto, re.M)
    if m:
        info["uc"] = m.group(1)
        info["uc_conces"] = int(m.group(1))
    m = re.search(r"^Cidade\s*:\s*(.+?)\s+-\s+", texto, re.M)
    if m:
        info["cidade"] = m.group(1).strip()
    m = re.search(r"^Bairro\s*:\s*(.+)$", texto, re.M)
    if m:
        info["bairro"] = m.group(1).strip()
    m = re.search(r"Utilizados\s*-\s*UC\s+(Geradora|Benefici)", texto, re.I)
    if m:
        info["tipo"] = "Geradora" if m.group(1).lower().startswith("gera") else "Beneficiária"
    m = re.search(r"originados da geradora UC\s*:\s*(\d+)", texto, re.I)
    if m:
        info["geradora_uc_conces"] = int(m.group(1))
    m = re.search(r"expirados no ciclo de faturamento\s*:\s*([\d.,]+)", texto, re.I)
    if m:
        info["expirados_kwh"] = limpar_numero(m.group(1))
    m = re.search(r"a expirar\s*:\s*([\d.,]+)\s*em\s*(\d{2}/\d{2}/\d{4})", texto, re.I)
    if m:
        info["prox_expirar_kwh"] = limpar_numero(m.group(1))
        info["prox_expirar_data"] = parse_data(m.group(2))
    m = re.search(r"Emitido em\s*:\s*(\d{2}/\d{2}/\d{4})", texto, re.I)
    if m:
        info["emitido_em"] = parse_data(m.group(1))
    return info

### 2.3 Tabela mapeada pelos rótulos

A linha de rótulos (`Referência`, `Saldo Anterior`, …) define onde cada métrica começa; ela ocupa
todas as colunas até o próximo rótulo. Os valores de todas as sub-colunas da métrica são **somados**
— com 1 sub-coluna (`TP`) é o próprio valor; com 2 (`PT`|`FP`) recupera o total. Assim o mesmo
código lê os dois layouts.

In [ ]:
CAMPOS = {                      # início do rótulo normalizado -> campo
    "saldo anterior":    "saldo_anterior",
    "cred. receb":       "cred_receb",
    "energia injetada":  "energia_injetada",
    "energia ativa":     "energia_ativa",
    "credito utilizado": "credito_utilizado",
    "percentual":        "percentual",
    "prioridade":        "prioridade",
    "saldo transferido": "saldo_transferido",
    "saldo final":       "saldo_final",
}
METRICAS = list(CAMPOS.values())


def _achar_tabela(page) -> list | None:
    for tabela in page.extract_tables():
        if any(c and "refer" in normalizar(c) for row in tabela[:3] for c in row):
            return tabela
    return None


def ler_tabela(page) -> list[dict]:
    tabela = _achar_tabela(page)
    if not tabela:
        return []
    i_hdr = next(i for i, row in enumerate(tabela) if any(c and "refer" in normalizar(c) for c in row))
    hdr = [normalizar(c) for c in tabela[i_hdr]]
    col_ref = next(j for j, c in enumerate(hdr) if "refer" in c)

    # spans: campo -> colunas [início, próximo rótulo)
    inicios = [j for j, c in enumerate(hdr) if c]
    spans = {}
    for k, j in enumerate(inicios):
        fim = inicios[k + 1] if k + 1 < len(inicios) else len(hdr)
        campo = next((v for r, v in CAMPOS.items() if hdr[j].startswith(r)), None)
        if campo:
            spans[campo] = range(j, fim)
    faltando = set(METRICAS) - set(spans)
    if faltando:
        raise ValueError(f"Rótulos não encontrados na tabela: {sorted(faltando)}")

    linhas = []
    for row in tabela[i_hdr + 1:]:
        ref = parse_referencia(row[col_ref] if col_ref < len(row) else None)
        if ref is None:
            continue                                   # linha de sub-cabeçalho (TP/PT/FP)
        d = {"referencia": ref}
        for campo, cols in spans.items():
            d[campo] = round(sum(limpar_numero(row[c]) for c in cols if c < len(row)), 4)
        linhas.append(d)
    return linhas

### 2.4 Orquestrador — `extrair_pdf()`

Devolve dois DataFrames: o histórico mensal de todas as seções e a situação de expiração de créditos
de cada seção.

In [ ]:
def nome_uc(uc_conces: int, cab: dict | None = None) -> str:
    if uc_conces in NOMES_UC:
        return NOMES_UC[uc_conces]
    local = "/".join(x for x in ((cab or {}).get("cidade"), (cab or {}).get("bairro")) if x)
    return f"UC {uc_conces}" + (f" - {local}" if local else "")


def extrair_pdf(pdf_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF não encontrado: {pdf_path.resolve()}")
    registros, expiracao = [], []
    with pdfplumber.open(str(pdf_path)) as pdf:
        logger.info("%s: %d página(s).", pdf_path.name, len(pdf.pages))
        for i, page in enumerate(pdf.pages, 1):
            texto = page.extract_text() or ""
            if not texto.strip():
                raise RuntimeError(f"{pdf_path.name} p.{i}: sem camada de texto (PDF escaneado?).")
            cab = ler_cabecalho(texto)
            if cab["uc_conces"] is None:
                logger.warning("%s p.%d: UC não encontrada — página ignorada.", pdf_path.name, i)
                continue
            linhas = ler_tabela(page)
            if not linhas:
                logger.warning("%s p.%d: nenhuma linha de tabela lida.", pdf_path.name, i)
                continue
            unidade = nome_uc(cab["uc_conces"], cab)
            for ln in linhas:
                registros.append(dict(
                    unidade=unidade, uc_conces=cab["uc_conces"], tipo=cab["tipo"],
                    geradora_uc_conces=cab["geradora_uc_conces"], **ln,
                    uc=cab["uc"], arquivo=pdf_path.name, emitido_em=cab["emitido_em"]))
            expiracao.append(dict(
                unidade=unidade, uc_conces=cab["uc_conces"], tipo=cab["tipo"],
                geradora_uc_conces=cab["geradora_uc_conces"],
                expirados_kwh=cab["expirados_kwh"], prox_expirar_kwh=cab["prox_expirar_kwh"],
                prox_expirar_data=cab["prox_expirar_data"], arquivo=pdf_path.name,
                emitido_em=cab["emitido_em"]))
            logger.info("  p.%d: %s (UC %s) — seção %s, origem %s, %d mês(es).", i, unidade,
                        cab["uc_conces"], cab["tipo"], cab["geradora_uc_conces"], len(linhas))

    df = pd.DataFrame(registros)
    # página repetida idêntica: mantém uma
    df = df.drop_duplicates(subset=CHAVE + METRICAS).reset_index(drop=True)
    # a mesma UC/origem pode ter 2+ quadros (ex.: troca de titularidade — um zerado e outro com os
    # dados, ou um com saldo antigo parado): os quadros são SOMADOS, não descartados
    if df.duplicated(subset=CHAVE).any():
        n_dup = int(df.duplicated(subset=CHAVE).sum())
        agg = {c: "sum" for c in METRICAS} | {"percentual": "max", "prioridade": "max"}
        agg |= {c: "first" for c in df.columns if c not in agg and c not in CHAVE}
        df = df.groupby(CHAVE, as_index=False, dropna=False, sort=False).agg(agg)[list(df.columns)]
        logger.info("%s: %d quadro(s) extra(s) da mesma UC/origem somado(s).", pdf_path.name, n_dup)
    exp = pd.DataFrame(expiracao)
    if not exp.empty:
        exp = exp.sort_values("prox_expirar_data", na_position="last")          # parcela mais próxima
        exp = exp.drop_duplicates(subset=["uc_conces", "geradora_uc_conces"])
    return df, exp.reset_index(drop=True)


CHAVE = ["uc_conces", "geradora_uc_conces", "referencia"]


def extrair_pdfs(paths: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Lê vários demonstrativos como UM só. A mesma seção (UC × origem × mês) presente em mais de
    um PDF — ex.: a Chácara como beneficiária da Nelson Souza — entra uma única vez (1º PDF da lista)."""
    dfs, exps = zip(*(extrair_pdf(Path(p)) for p in paths))
    df = pd.concat(dfs, ignore_index=True)
    n0 = len(df)
    df = df.drop_duplicates(subset=CHAVE, keep="first").reset_index(drop=True)
    if n0 > len(df):
        logger.info("%d linha(s) repetida(s) entre os demonstrativos descartada(s).", n0 - len(df))
    exp = pd.concat(exps, ignore_index=True)
    if not exp.empty:
        exp = (exp.sort_values("prox_expirar_data", na_position="last")
                  .drop_duplicates(subset=["uc_conces", "geradora_uc_conces"]).reset_index(drop=True))
    return df, exp

## 3. Validação dos números lidos

Duas conferências por seção (UC × origem):

1. **Balanço do mês:** `saldo_final = saldo_anterior + cred_receb + energia_injetada − credito_utilizado − saldo_transferido`
2. **Continuidade:** `saldo_final` de um mês = `saldo_anterior` do mês seguinte.

Divergências são apenas **avisadas** (podem vir de créditos expirados ou de ajustes da Copel), mas
costumam indicar erro de leitura.

In [ ]:
def validar(df: pd.DataFrame, tolerancia: float = 1.0) -> pd.DataFrame:
    problemas = []
    calc = (df["saldo_anterior"] + df["cred_receb"] + df["energia_injetada"]
            - df["credito_utilizado"] - df["saldo_transferido"])
    for idx in df.index[(calc - df["saldo_final"]).abs() > tolerancia]:
        r = df.loc[idx]
        problemas.append(dict(unidade=r["unidade"], origem=r["geradora_uc_conces"],
                              referencia=r["referencia"], conferencia="balanço do mês",
                              esperado=calc[idx], lido=r["saldo_final"]))
    for _, g in df.sort_values("referencia").groupby(["uc_conces", "geradora_uc_conces"]):
        g = g.reset_index(drop=True)
        for k in range(1, len(g)):
            meses = (g.at[k, "referencia"].year - g.at[k - 1, "referencia"].year) * 12 \
                    + g.at[k, "referencia"].month - g.at[k - 1, "referencia"].month
            if meses == 1 and abs(g.at[k, "saldo_anterior"] - g.at[k - 1, "saldo_final"]) > tolerancia:
                problemas.append(dict(unidade=g.at[k, "unidade"], origem=g.at[k, "geradora_uc_conces"],
                                      referencia=g.at[k, "referencia"], conferencia="continuidade",
                                      esperado=g.at[k - 1, "saldo_final"], lido=g.at[k, "saldo_anterior"]))
    res = pd.DataFrame(problemas)
    if res.empty:
        logger.info("Validação OK: %d linha(s), balanço e continuidade conferem.", len(df))
    else:
        logger.warning("Validação: %d divergência(s) — veja a tabela.", len(res))
    return res

## 4. Planilha

### 4.1 Base consolidada (histórico existente + PDF)

- Se a planilha de saída já existir, a aba `Demonstrativo` é lida e mesclada com o PDF.
- `consumo` = *Energia Ativa* registrada **uma vez** por UC/mês (na seção *Geradora*, se houver;
  senão na primeira seção). É a coluna usada nas fórmulas.

In [ ]:
COLS_MESTRE = ["unidade", "uc_conces", "tipo", "geradora_uc_conces", "referencia", "consumo",
               "energia_ativa", "saldo_anterior", "cred_receb", "energia_injetada",
               "credito_utilizado", "percentual", "prioridade", "saldo_transferido",
               "saldo_final", "uc", "arquivo", "emitido_em"]
TITULOS_MESTRE = {
    "unidade": "Unidade", "uc_conces": "UC", "tipo": "Seção",
    "geradora_uc_conces": "Geradora de origem (UC)", "referencia": "Referência",
    "consumo": "Consumo (kWh)", "energia_ativa": "Energia Ativa no PDF (kWh)",
    "saldo_anterior": "Saldo Anterior (kWh)", "cred_receb": "Créd. Receb. Outra UC (kWh)",
    "energia_injetada": "Energia Injetada (kWh)", "credito_utilizado": "Crédito Utilizado (kWh)",
    "percentual": "Percentual (%)", "prioridade": "Prioridade",
    "saldo_transferido": "Saldo Transf. Outra UC (kWh)", "saldo_final": "Saldo Final (kWh)",
    "uc": "UC (texto do PDF)", "arquivo": "Arquivo", "emitido_em": "Emitido em",
}
L = {c: get_column_letter(i + 1) for i, c in enumerate(COLS_MESTRE)}   # campo -> letra
ABA_MESTRE = "Demonstrativo"


def secoes_relatorio(base: pd.DataFrame) -> list:
    """Uma linha por seção (UC × geradora de origem) — o saldo NÃO é somado entre origens.
    Seção Geradora recebe o sufixo TAG_GERADORA. Ordem: beneficiárias por nome, depois geradoras."""
    sec = (base.groupby(["uc_conces", "geradora_uc_conces", "tipo"], dropna=False)["unidade"]
               .last().reset_index())
    out = []
    for r in sec.itertuples(index=False):
        ger = r.tipo == "Geradora"
        out.append(dict(rotulo=f"{r.unidade} {TAG_GERADORA}" if ger else str(r.unidade),
                        unidade=r.unidade, uc=int(r.uc_conces), tipo=r.tipo, geradora=ger,
                        origem=int(r.geradora_uc_conces) if pd.notna(r.geradora_uc_conces) else None))
    return sorted(out, key=lambda d: (d["geradora"], d["rotulo"]))
ABA_EXPIRA = "Créditos a Expirar"


def carregar_wb(path: Path):
    # BytesIO: o openpyxl não prende o arquivo no Windows
    return openpyxl.load_workbook(io.BytesIO(path.read_bytes()), data_only=False)


def ler_base_existente(xlsx: Path) -> pd.DataFrame:
    if not xlsx.exists():
        return pd.DataFrame(columns=COLS_MESTRE)
    wb = carregar_wb(xlsx)
    if ABA_MESTRE not in wb.sheetnames:
        logger.warning("%s existe mas não tem a aba %r — será recriada do zero.", xlsx.name, ABA_MESTRE)
        return pd.DataFrame(columns=COLS_MESTRE)
    linhas = list(wb[ABA_MESTRE].iter_rows(min_row=2, max_col=len(COLS_MESTRE), values_only=True))
    df = pd.DataFrame([r for r in linhas if r[1] is not None], columns=COLS_MESTRE)
    logger.info("%s: %d linha(s) de histórico existente.", xlsx.name, len(df))
    return df


def montar_base(df_pdf: pd.DataFrame, df_existente: pd.DataFrame) -> pd.DataFrame:
    partes = [df_pdf] + ([df_existente] if not df_existente.empty else [])
    base = pd.concat(partes, ignore_index=True)                          # PDF primeiro = prevalece
    base["geradora_uc_conces"] = base["geradora_uc_conces"].astype("Int64")
    base["uc_conces"] = base["uc_conces"].astype("int64")
    base["referencia"] = pd.to_datetime(base["referencia"])
    base = base.drop_duplicates(subset=CHAVE, keep="first")
    # nome atual do mapa (renomear em NOMES_UC reflete no histórico antigo também)
    base["unidade"] = [NOMES_UC.get(uc, u) for uc, u in zip(base["uc_conces"], base["unidade"])]

    # consumo único por UC/mês: seção Geradora primeiro
    base["_ordem"] = (base["tipo"] != "Geradora").astype("int64")
    base = base.sort_values(["uc_conces", "referencia", "_ordem", "geradora_uc_conces"])
    primeira = ~base.duplicated(subset=["uc_conces", "referencia"])
    base["consumo"] = base["energia_ativa"].where(primeira, 0.0)
    logger.info("Base consolidada: %d linha(s) (%d do histórico já existente).",
                len(base), len(df_existente))
    return (base.sort_values(["referencia", "unidade", "_ordem", "geradora_uc_conces"])
                .reset_index(drop=True)[COLS_MESTRE])

### 4.2 Gravação do workbook

Abas geradas (nesta ordem):

| Aba | Conteúdo |
|---|---|
| `Resumo Ultimo Mês` | 1 linha por seção (UC × origem) no mês mais recente (`B1 = MAX(Demonstrativo!E:E)` — atualiza sozinho) |
| `Resumo Anual (AAAA)` | acumulado do ano do mês mais recente; saldo = saldo do último mês |
| 1 aba por seção | histórico mensal com fórmulas `SUMIFS` (UC + origem) sobre o `Demonstrativo` + linha `TOTAL` |
| `Demonstrativo` | aba-mestre com os dados brutos (1 linha por UC × origem × mês) |
| `Créditos a Expirar` | expiração por seção, conforme o demonstrativo mais recente |

As fórmulas ganham valor quando o Excel abrir o arquivo (ele recalcula ao abrir).

In [ ]:
COR_HDR   = "0F3D3E"
FILL_HDR  = PatternFill("solid", fgColor="CCCCCC")   # padrão das planilhas PR/SC
FILL_TOT  = PatternFill("solid", fgColor="CCCCCC")
FONT_HDR  = Font(bold=True, color="000000")
FONT_TIT  = Font(bold=True, size=13, color="000000")
BORDA     = Border(bottom=Side(style="thin", color="C9D3D4"))
FMT_KWH   = '#,##0;-#,##0;"-"'
FMT_PCT   = '0.0%'
FMT_MES   = 'mm/yyyy'
FMT_DATA  = 'dd/mm/yyyy'
FMT_UC    = '000000000000000'                  # UC com 15 dígitos, como no PDF


def _cabecalho(ws, linha: int, titulos: list, larguras: list | None = None):
    for j, t in enumerate(titulos, 1):
        c = ws.cell(linha, j, t)
        c.fill, c.font = FILL_HDR, FONT_HDR
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[linha].height = 32
    for j, w in enumerate(larguras or [], 1):
        ws.column_dimensions[get_column_letter(j)].width = w


def _crit(uc_ref: str, ini: str, fim: str, ger_ref: str | None = None) -> str:
    """Critérios SUMIFS: UC (+ geradora de origem) + intervalo de datas no Demonstrativo."""
    d = f"{ABA_MESTRE}!"
    origem = (f'{d}{L["geradora_uc_conces"]}:{L["geradora_uc_conces"]}, {ger_ref}, ' if ger_ref else "")
    uc = f'{d}{L["uc_conces"]}:{L["uc_conces"]}, {uc_ref}, ' if uc_ref else ""
    return (uc + origem +
            f'{d}{L["referencia"]}:{L["referencia"]}, ">="&{ini}, '
            f'{d}{L["referencia"]}:{L["referencia"]}, "<="&{fim}')


def _soma(campo: str, uc_ref: str, ini: str, fim: str, ger_ref: str | None = None) -> str:
    return f'=SUMIFS({ABA_MESTRE}!{L[campo]}:{L[campo]}, {_crit(uc_ref, ini, fim, ger_ref)})'


# colunas de fluxo exibidas nas abas de UC e nos resumos: (título, campo)
FLUXOS = [("Consumo (kWh)", "energia_ativa"),      # consumo da seção (como no PDF)
          ("Energia Injetada (kWh)", "energia_injetada"),
          ("Créd. Recebido Outra UC (kWh)", "cred_receb"),
          ("Crédito Utilizado (kWh)", "credito_utilizado"),
          ("Saldo Transf. Outra UC (kWh)", "saldo_transferido"),
          ("Saldo Final (kWh)", "saldo_final")]


def _nome_aba(nome: str, usados: set) -> str:
    base = re.sub(r"[\[\]\*\?/\\:]", "-", nome)[:31]
    nome_final, k = base, 2
    while nome_final in usados:
        sufixo = f" ({k})"
        nome_final, k = base[:31 - len(sufixo)] + sufixo, k + 1
    usados.add(nome_final)
    return nome_final


def escrever_mestre(ws, base: pd.DataFrame):
    _cabecalho(ws, 1, [TITULOS_MESTRE[c] for c in COLS_MESTRE],
               [18, 17, 13, 18, 11, 12, 13, 13, 14, 13, 13, 11, 10, 14, 13, 17, 34, 11])
    for i, row in enumerate(base.itertuples(index=False), 2):
        for j, campo in enumerate(COLS_MESTRE, 1):
            v = getattr(row, campo)
            if pd.isna(v):
                v = None
            elif isinstance(v, pd.Timestamp):
                v = v.to_pydatetime()
            elif hasattr(v, "item"):
                v = v.item()                          # numpy -> python
            c = ws.cell(i, j, v)
            if campo == "referencia":
                c.number_format = FMT_MES
            elif campo == "emitido_em":
                c.number_format = FMT_DATA
            elif campo in ("uc_conces", "geradora_uc_conces"):
                c.number_format = FMT_UC
            elif campo not in ("unidade", "uc_conces", "tipo", "geradora_uc_conces", "uc",
                               "arquivo", "percentual", "prioridade"):
                c.number_format = FMT_KWH
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(COLS_MESTRE))}{max(len(base) + 1, 2)}"


def escrever_aba_uc(ws, sec: dict, meses: list):
    """Aba de UMA seção (UC × geradora de origem): saldo só desta seção, como no PDF."""
    ws["A1"] = sec["rotulo"]; ws["A1"].font = FONT_TIT
    ws["A2"] = "UC:";     ws["B2"] = sec["uc"];     ws["B2"].number_format = FMT_UC
    ws["C2"] = "Origem:"; ws["D2"] = sec["origem"]; ws["D2"].number_format = FMT_UC
    ws["E2"] = sec["tipo"]
    for c in ("A2", "C2"):
        ws[c].font = Font(bold=True)
    _cabecalho(ws, 3, ["Mês/Ano"] + [t for t, _ in FLUXOS] + ["Cobertura %"],
               [11, 17, 15, 17, 17, 17, 14, 12])
    r0 = 4
    for i, mes in enumerate(meses):
        r = r0 + i
        ws.cell(r, 1, mes.to_pydatetime()).number_format = FMT_MES
        for j, (_, campo) in enumerate(FLUXOS, 2):
            ws.cell(r, j, _soma(campo, "$B$2", f"$A{r}", f"EOMONTH($A{r},0)", "$D$2")).number_format = FMT_KWH
        ws.cell(r, len(FLUXOS) + 2, f'=IFERROR(E{r}/B{r},"")').number_format = FMT_PCT
        for j in range(1, len(FLUXOS) + 3):
            ws.cell(r, j).border = BORDA
    ult = r0 + len(meses) - 1
    t = ult + 1
    ws.cell(t, 1, "TOTAL")
    col_saldo = get_column_letter(len(FLUXOS) + 1)
    for j in range(2, len(FLUXOS) + 2):
        L_ = get_column_letter(j)
        f = f"={col_saldo}{ult}" if L_ == col_saldo else f"=SUM({L_}{r0}:{L_}{ult})"
        ws.cell(t, j, f).number_format = FMT_KWH
    ws.cell(t, len(FLUXOS) + 2, f'=IFERROR(E{t}/B{t},"")').number_format = FMT_PCT
    for j in range(1, len(FLUXOS) + 3):
        ws.cell(t, j).fill, ws.cell(t, j).font = FILL_TOT, Font(bold=True)
    ws.cell(t + 1, 1, "Saldo Final no TOTAL = saldo do último mês desta seção (origem acima). "
                      "Cobertura % = crédito utilizado ÷ consumo.").font = Font(italic=True, size=9)
    ws.freeze_panes = "B4"


def escrever_resumo(ws, secoes: list, anual: bool):
    """1 linha por seção (UC × origem) — o saldo não é somado entre origens."""
    ws["A1"] = "Mês de referência:"; ws["A1"].font = Font(bold=True)
    ws["B1"] = f"=MAX({ABA_MESTRE}!{L['referencia']}:{L['referencia']})"
    ws["B1"].number_format = FMT_MES
    if anual:
        ws["C1"] = "Ano:"; ws["C1"].font = Font(bold=True)
        ws["D1"] = "=YEAR(B1)"
        ini, fim = "DATE($D$1,1,1)", "EOMONTH($B$1,0)"
        titulo = "Acumulado do ano (saldo = posição no mês de referência)"
    else:
        ini, fim = "$B$1", "EOMONTH($B$1,0)"
        titulo = "Último mês"
    ws["A2"] = titulo; ws["A2"].font = Font(italic=True)
    _cabecalho(ws, 3, ["Unidade", "UC", "Origem (UC geradora)"] + [t for t, _ in FLUXOS] + ["Cobertura %"],
               [40, 17, 18, 14, 15, 17, 15, 17, 14, 12])
    c0 = 4                                             # 1ª coluna de fluxo
    col = {campo: get_column_letter(c0 + k) for k, (_, campo) in enumerate(FLUXOS)}
    n_cols = c0 + len(FLUXOS)                          # última coluna (Cobertura %)
    r0 = 4
    for i, sec in enumerate(secoes):
        r = r0 + i
        ws.cell(r, 1, sec["rotulo"])
        ws.cell(r, 2, sec["uc"]).number_format = FMT_UC
        ws.cell(r, 3, sec["origem"]).number_format = FMT_UC
        for j, (_, campo) in enumerate(FLUXOS, c0):
            a, b = (("$B$1", "EOMONTH($B$1,0)") if campo == "saldo_final" else (ini, fim))
            ws.cell(r, j, _soma(campo, f"$B{r}", a, b, f"$C{r}")).number_format = FMT_KWH
        ws.cell(r, n_cols, f'=IFERROR({col["credito_utilizado"]}{r}/{col["energia_ativa"]}{r},"")').number_format = FMT_PCT
        for j in range(1, n_cols + 1):
            ws.cell(r, j).border = BORDA
    ult = r0 + len(secoes) - 1
    t = ult + 1
    ws.cell(t, 1, "TOTAL")
    for j, (_, campo) in enumerate(FLUXOS, c0):
        L_ = get_column_letter(j)
        if campo == "energia_ativa":                   # consumo de cada UC contado UMA vez
            f = _soma("consumo", None, ini, fim)
        else:
            f = f"=SUM({L_}{r0}:{L_}{ult})"
        ws.cell(t, j, f).number_format = FMT_KWH
    ws.cell(t, n_cols, f'=IFERROR({col["credito_utilizado"]}{t}/{col["energia_ativa"]}{t},"")').number_format = FMT_PCT
    for j in range(1, n_cols + 1):
        ws.cell(t, j).fill, ws.cell(t, j).font = FILL_TOT, Font(bold=True)
    ws.cell(t + 1, 1, "Uma linha por seção do demonstrativo (UC × geradora de origem); o saldo não é somado "
                      "entre origens. Consumo do TOTAL conta cada UC uma vez.").font = Font(italic=True, size=9)
    ws.freeze_panes = "A4"


def escrever_expiracao(ws, exp: pd.DataFrame):
    cols = [("Unidade", "unidade", 30), ("UC", "uc_conces", 17), ("Seção", "tipo", 13),
            ("Geradora de origem (UC)", "geradora_uc_conces", 18),
            ("Expirados no ciclo (kWh)", "expirados_kwh", 15),
            ("Próxima parcela a expirar (kWh)", "prox_expirar_kwh", 17),
            ("Data da expiração", "prox_expirar_data", 14), ("Emitido em", "emitido_em", 12),
            ("Arquivo", "arquivo", 34)]
    _cabecalho(ws, 1, [c[0] for c in cols], [c[2] for c in cols])
    for i, row in enumerate(exp.to_dict("records"), 2):
        for j, (_, campo, _) in enumerate(cols, 1):
            v = row[campo]
            c = ws.cell(i, j, None if pd.isna(v) else v)
            if campo in ("prox_expirar_data", "emitido_em"):
                c.number_format = FMT_DATA
            elif campo.endswith("_kwh"):
                c.number_format = FMT_KWH
            elif campo in ("uc_conces", "geradora_uc_conces"):
                c.number_format = FMT_UC
    ws.freeze_panes = "A2"


def escrever_cadastro(ws, xlsx: Path = CADASTRO_XLSX, aba: str = ABA_CADASTRO):
    """Copia a aba `Unidades Consumidoras` do cadastro para a planilha de saída (só valores)."""
    wb_c = openpyxl.load_workbook(io.BytesIO(xlsx.read_bytes()), data_only=True)
    ws_c = wb_c[aba]
    for row in ws_c.iter_rows():
        for c in row:
            if c.value is not None:
                ws.cell(c.row, c.column, c.value)
    for letra, dim in ws_c.column_dimensions.items():
        if dim.width:
            ws.column_dimensions[letra].width = dim.width
    for c in ws[1]:
        if c.value is not None:
            c.fill, c.font = FILL_HDR, FONT_HDR


def gravar_planilha(base: pd.DataFrame, exp: pd.DataFrame, xlsx: Path) -> Path:
    wb = openpyxl.Workbook()
    wb.remove(wb.active)
    ano = base["referencia"].max().year

    secoes = secoes_relatorio(base)               # 1 linha/aba por seção (UC × origem)

    usados = {ABA_MESTRE, ABA_EXPIRA, ABA_CADASTRO, "Resumo Ultimo Mês", f"Resumo Anual ({ano})"}
    escrever_resumo(wb.create_sheet("Resumo Ultimo Mês"), secoes, anual=False)
    escrever_resumo(wb.create_sheet(f"Resumo Anual ({ano})"), secoes, anual=True)
    for sec in secoes:
        m = (base["uc_conces"] == sec["uc"]) & (base["geradora_uc_conces"] == sec["origem"])
        meses = sorted(base.loc[m, "referencia"].unique())
        escrever_aba_uc(wb.create_sheet(_nome_aba(sec["rotulo"], usados)), sec,
                        [pd.Timestamp(x) for x in meses])
    escrever_mestre(wb.create_sheet(ABA_MESTRE), base)
    escrever_expiracao(wb.create_sheet(ABA_EXPIRA), exp)
    if CADASTRO_XLSX.exists():
        try:
            escrever_cadastro(wb.create_sheet(ABA_CADASTRO))
        except KeyError:
            del wb[ABA_CADASTRO]                  # cadastro sem a aba: segue sem ela

    for ws in wb.worksheets:                      # abas-base em laranja, como nas planilhas PR/SC
        if ws.title in (ABA_MESTRE, ABA_EXPIRA, ABA_CADASTRO) or ws.title.startswith("Resumo"):
            ws.sheet_properties.tabColor = "FF9900"
    buf = io.BytesIO()
    wb.save(buf)
    xlsx.write_bytes(buf.getvalue())
    logger.info("Planilha salva: %s (%d aba(s)).", xlsx, len(wb.sheetnames))
    return xlsx

### 4.3 Resumo calculado em Python

Mesmo cálculo das abas de resumo, feito direto na base — útil para conferir o resultado sem abrir
o Excel.

In [ ]:
def resumo_python(base: pd.DataFrame, mes=None) -> pd.DataFrame:
    mes = pd.Timestamp(mes) if mes is not None else base["referencia"].max()
    campos = [c for _, c in FLUXOS]
    r = (base[base["referencia"] == mes]
             .groupby(["unidade", "tipo", "uc_conces", "geradora_uc_conces"])[campos].sum().reset_index())
    r["cobertura_%"] = (r["credito_utilizado"] / r["energia_ativa"]).where(r["energia_ativa"] > 0).mul(100).round(1)
    r.insert(0, "mes", mes.strftime("%m/%Y"))
    return r

## 5. Painel executivo (PNG/PDF) — padrão PR

Mesmo layout do painel do notebook Copel PR:

- **KPIs** (só as beneficiárias da geradora do rateio): consumo total, energia injetada (créditos recebidos),
  cobertura média, UCs com superávit e UCs críticas.
- **Tabela**: uma linha por **seção** do demonstrativo (UC × geradora de origem) — o saldo **não é somado**
  entre origens. Seções Geradora levam o sufixo `(GERADORA)`. Colunas: Consumida, Injetada, Média Inj.,
  Saldo, % Atual, % Ideal (▲▼), Inj. p/ Ideal (~kWh) e o *chip* de status.
- **Rodapé**: *Atenção urgente — reforçar alocação* e *Superávit acumulado — reduzir alocação*.

| Status | Condição | Cor |
|---|---|---|
| **Superávit** | `saldo > LIM_SUPERAVIT` | verde |
| **Confortável** | `LIM_CONFORTAVEL < saldo ≤ LIM_SUPERAVIT` | azul |
| **Saldo baixo** | `LIM_BAIXO ≤ saldo ≤ LIM_CONFORTAVEL` | âmbar |
| **Crítico / Zero** | `saldo < LIM_BAIXO` | vermelho |
| **Saldo preso** | saldo parado (rateio anual) e utilizável ≤ `LIM_CONFORTAVEL` | roxo |

A **% Ideal** vem do rateio anual da geradora com beneficiárias (Nelson Souza). As funções do painel
antigo (5.1 e 5.2) continuam no notebook mas não são usadas; o painel padrão PR está na célula 5.3.

### 5.1 Agregação, status e % ideal (painel antigo)


In [ ]:
def _meses_entre(a, b) -> int:
    return (b.year - a.year) * 12 + b.month - a.month


def classificar(saldo: float, autonomia: float | None):
    """(rótulo, cor) do status do saldo pela autonomia em meses."""
    if saldo < LIM_BAIXO:
        return "Crítico / Zero", "#b23b3b"
    if autonomia is None or autonomia > LIM_MESES_SUPERAVIT:
        return "Superávit", "#1a7f5a"
    if autonomia > LIM_MESES_CONFORTAVEL:
        return "Confortável", "#2f7fb0"
    return "Saldo baixo", "#c98a1a"


def _aplicar_piso(pesos: dict, total: float, piso: float) -> dict:
    """Reparte `total` proporcional a `pesos`, garantindo `piso` a cada um (soma conservada)."""
    fixos, livres = {}, dict(pesos)
    while True:
        resto = total - piso * len(fixos)
        soma = sum(livres.values())
        if not livres or resto <= 0:
            return {k: total / max(len(pesos), 1) for k in pesos}
        parc = {k: (resto * w / soma if soma else resto / len(livres)) for k, w in livres.items()}
        abaixo = [k for k, v in parc.items() if v < piso]
        if not piso or not abaixo:
            return {**{k: piso for k in fixos}, **parc}
        for k in abaixo:
            fixos[k] = piso
            livres.pop(k)


def _arredondar_conservando(valores: dict, casas: int = 2) -> dict:
    """Arredonda mantendo a soma (maior resto) — a soma digitada no site da Copel fecha exata."""
    q = 10 ** casas
    alvo = round(sum(valores.values()) * q)
    base = {k: int(v * q) for k, v in valores.items()}
    falta = alvo - sum(base.values())
    for k in sorted(valores, key=lambda k: valores[k] * q - base[k], reverse=True)[:max(falta, 0)]:
        base[k] += 1
    return {k: v / q for k, v in base.items()}


def geradoras_ordenadas(base: pd.DataFrame) -> list:
    """UCs com seção Geradora; primeiro a que tem mais beneficiárias (é a do rateio do painel)."""
    ger = base.loc[base["tipo"] == "Geradora", "uc_conces"].astype("int64").unique()
    benef = base[base["tipo"] == "Beneficiária"]
    n = {int(g): benef.loc[benef["geradora_uc_conces"] == g, "uc_conces"].nunique() for g in ger}
    return sorted(n, key=lambda g: (-n[g], nome_uc(g)))


def agregar_painel(base: pd.DataFrame, rateio_w: dict | None = None) -> dict:
    """Números do painel a partir da base consolidada (aba Demonstrativo)."""
    base = base.copy()
    base["referencia"] = pd.to_datetime(base["referencia"])
    mes = base["referencia"].max()

    # série mensal (todas as UCs do demonstrativo)
    serie = (base.groupby("referencia")[["energia_injetada", "consumo", "saldo_final"]].sum()
                 .sort_index())
    serie = serie[serie.index > mes - pd.DateOffset(months=13)]

    # geradora do rateio do painel = a que tem beneficiárias; todas as geradoras ficam em destaque
    geradoras = geradoras_ordenadas(base)
    geradora = geradoras[0] if geradoras else None

    ucs = []
    for uc, g in base.groupby("uc_conces"):
        g_mes = g[g["referencia"] == mes]
        if g_mes.empty:
            continue
        cons_mes = g.groupby("referencia")["consumo"].sum().sort_index()
        janela = cons_mes.iloc[-int(JANELA_MEDIA_MESES):] if JANELA_MEDIA_MESES else cons_mes
        consumo_medio = float(janela.mean()) if len(janela) else 0.0
        saldo = float(g_mes["saldo_final"].sum())
        benef = g_mes[(g_mes["tipo"] == "Beneficiária") & (g_mes["geradora_uc_conces"] == geradora)]
        ucs.append(dict(
            uc_conces=int(uc), unidade=g_mes["unidade"].iloc[0],
            secoes=" + ".join(sorted(set(g["tipo"].dropna()), reverse=True)),
            eh_geradora=int(uc) in geradoras,
            consumo=float(g_mes["consumo"].sum()),
            injetada=float(g_mes["energia_injetada"].sum()),
            recebido=float(g_mes["cred_receb"].sum()),
            utilizado=float(g_mes["credito_utilizado"].sum()),
            transferido=float(g_mes["saldo_transferido"].sum()),
            saldo=saldo, consumo_medio=consumo_medio,
            autonomia=(saldo / consumo_medio) if consumo_medio > 0 else None,
            pct_atual=float(benef["percentual"].sum()) if not benef.empty else None,
        ))
    for d in ucs:
        d["status"], d["cor"] = classificar(d["saldo"], d["autonomia"])

    # variação do saldo total: média mensal nos últimos 12 meses
    saldos = serie["saldo_final"]
    ref_ant = saldos.index[0]
    var_mensal = ((saldos.iloc[-1] - saldos.iloc[0]) / _meses_entre(ref_ant, saldos.index[-1])
                  if len(saldos) > 1 else 0.0)

    painel = dict(mes=mes, serie=serie, geradora=geradora, ucs=ucs, var_mensal=var_mensal,
                  rateio=sugerir_pct_ideal(base, ucs, geradora, mes))
    # (v2) % ideal do rateio anual (Etapa 4), quando disponível
    if ESTRATEGIA_RATEIO == "W" and rateio_w and painel["rateio"].get("ativo"):
        tab, fator = rateio_w["tabela"], painel["rateio"]["fator"]
        for d in ucs:
            if d["pct_atual"] is not None and d["unidade"] in tab.index:
                d["pct_ideal"] = float(tab.loc[d["unidade"], "pct_ideal"])
                d["delta_kwh"] = round((d["pct_ideal"] - d["pct_atual"]) * fator)
        painel["rateio"]["estrategia"] = "W"
    # status pelo saldo UTILIZÁVEL: saldo preso em cadastro antigo não compensa a fatura
    if rateio_w:
        preso = rateio_w["tabela"]["saldo_preso"]
        for d in ucs:
            d["preso"] = float(preso.get(d["unidade"], 0.0))
            if d["preso"] > 0 and not d["eh_geradora"]:
                util = d["saldo"] - d["preso"]
                aut = (util / d["consumo_medio"]) if d["consumo_medio"] > 0 else None
                d["status"], d["cor"] = classificar(util, aut)
                if aut is not None and aut <= LIM_MESES_CONFORTAVEL:
                    d["status"], d["cor"] = "Saldo preso", "#6b5b95"
    return painel


def sugerir_pct_ideal(base: pd.DataFrame, ucs: list, geradora: int | None, mes) -> dict:
    """% ideal do rateio da geradora entre as suas beneficiárias (Σ% conservada)."""
    benef = [d for d in ucs if d["pct_atual"] is not None]
    for d in ucs:
        d["pct_ideal"] = d["delta_kwh"] = None
    if geradora is None or not benef:
        return dict(ativo=False)
    total_pct = sum(d["pct_atual"] for d in benef)

    # fator kWh por 1%: média dos últimos 12 meses de (Σ recebido da geradora ÷ Σ %)
    s = base[(base["tipo"] == "Beneficiária") & (base["geradora_uc_conces"] == geradora)]
    s = s[s["referencia"] > mes - pd.DateOffset(months=12)]
    por_mes = s.groupby("referencia")[["cred_receb", "percentual"]].sum()
    por_mes = por_mes[por_mes["percentual"] > 0]
    fator = float((por_mes["cred_receb"] / por_mes["percentual"]).mean()) if len(por_mes) else 0.0

    estrategia = "C" if CONSIDERAR_SALDO else "A"
    pesos = {d["uc_conces"]: d["consumo_medio"] for d in benef}
    if CONSIDERAR_SALDO:
        meses = HORIZONTE_MESES + MESES_BUFFER
        nec = {d["uc_conces"]: max(0.0, meses * d["consumo_medio"] - d["saldo"]) for d in benef}
        if sum(nec.values()) > 0:
            pesos = nec
        else:
            logger.warning("Estratégia C: nenhuma beneficiária com necessidade — usando A.")
            estrategia = "A"
    alocado = _arredondar_conservando(_aplicar_piso(pesos, total_pct, PISO_PCT))
    for d in benef:
        d["pct_ideal"] = alocado[d["uc_conces"]]
        d["delta_kwh"] = round((d["pct_ideal"] - d["pct_atual"]) * fator)
    logger.info("%% ideal | estratégia %s: %s", estrategia,
                ", ".join(f'{d["unidade"]} {d["pct_atual"]:.2f}→{d["pct_ideal"]:.2f}' for d in benef))
    return dict(ativo=True, estrategia=estrategia, total_pct=total_pct, fator=fator)

### 5.2 Renderização (`matplotlib`)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

COR_FUNDO   = "#f4f6f8"
COR_HEADER  = "#0f3d3e"
COR_HEADER2 = "#155e63"
COR_TXT     = "#1c2b2b"
COR_MUTED   = "#5a6b6b"
COR_LINHA_A = "#ffffff"
COR_LINHA_B = "#eef2f3"
COR_GERACAO = "#2a78d6"     # série 1 (azul)
COR_CONSUMO = "#eb6834"     # série 2 (laranja)
COR_SALDO   = COR_HEADER2   # série única no gráfico de saldo
MESES_BR = {1: "jan", 2: "fev", 3: "mar", 4: "abr", 5: "mai", 6: "jun",
            7: "jul", 8: "ago", 9: "set", 10: "out", 11: "nov", 12: "dez"}


def _fmt(n) -> str:
    try:
        return f"{float(n):,.0f}".replace(",", ".")
    except (TypeError, ValueError):
        return str(n)


def _fmt_sinal(n) -> str:
    n = round(float(n))
    return ("+" if n > 0 else ("−" if n < 0 else "±")) + _fmt(abs(n))


def _fmt_pct(v) -> str:
    return "—" if v is None else f"{v:.2f}".rstrip("0").rstrip(".")


def _mes_curto(ts) -> str:
    return f"{MESES_BR[ts.month]}/{str(ts.year)[2:]}"


def _estilo_eixo(ax):
    ax.set_facecolor("white")
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#c9d3d4")
    ax.tick_params(axis="both", labelsize=7.5, colors=COR_MUTED, length=0)
    ax.grid(axis="y", color="#e3e9ea", lw=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: _fmt(v)))


def gerar_painel(nome: str, painel: dict, exp: pd.DataFrame, saida_base: Path):
    ucs = sorted(painel["ucs"], key=lambda d: (not d["eh_geradora"], d["unidade"]))
    mes, serie, rateio = painel["mes"], painel["serie"], painel["rateio"]
    var = painel["var_mensal"]

    # texto da caixa de rateio (montado antes: a altura das caixas depende do nº de linhas)
    ger = next((d for d in ucs if d["uc_conces"] == painel["geradora"]), None)
    titulo = f"⚙  Rateio da geradora {ger['unidade'] if ger else ''}".rstrip()
    linhas, nota = [], ""
    if rateio.get("ativo"):
        for d in [u for u in ucs if u["pct_ideal"] is not None]:
            linhas.append(f'{d["unidade"]}:  {_fmt_pct(d["pct_atual"])}% → {_fmt_pct(d["pct_ideal"])}%'
                          f'  ({_fmt_sinal(d["delta_kwh"])} kWh/mês)')
        nota = (("Estratégia W: cobertura anual equalizada"
                 if rateio["estrategia"] == "W" else
                 f"Estratégia {rateio['estrategia']}, horizonte {HORIZONTE_MESES} meses")
                + (f" (+{MESES_BUFFER:.1f} de margem), descontando o saldo" if rateio["estrategia"] == "C" else "")
                + f".\nΣ% conservada ({_fmt_pct(rateio['total_pct'])}%); ~{rateio['fator']:.1f} kWh por 1% "
                  f"(média de 12 meses).")
    elif ger:
        linhas.append("Sem beneficiárias: toda a geração fica na própria UC.")
        if ger["autonomia"] and ger["autonomia"] > VALIDADE_CREDITOS_MESES:
            linhas.append(f'O saldo cobre ~{ger["autonomia"]:.0f} meses de consumo — mais que a '
                          f"validade de {VALIDADE_CREDITOS_MESES} meses dos créditos.")
            linhas.append("Avaliar incluir beneficiárias para não perder créditos.")
        nota = f"Saldo nos últimos 12 meses: {_fmt_sinal(var)} kWh/mês."
    else:
        linhas.append("Nenhuma seção Geradora neste demonstrativo.")

    n = len(ucs)
    row_h = 0.34
    y_graf, h_graf = 2.75, 2.55                   # bloco dos gráficos (polegadas a partir do topo)
    yh = y_graf + h_graf + 0.45                   # cabeçalho da tabela
    fy = yh + 0.5 + n * row_h + 0.35              # caixas do rodapé
    box_h = 1.1 + 0.26 * max(len(linhas), len(exp), 2)
    H = fy + box_h + 0.85
    fig = plt.figure(figsize=(13.6, H), dpi=140)
    fig.patch.set_facecolor(COR_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    def y(pol): return 1 - pol / H

    # ----- cabeçalho -----
    ax.add_patch(FancyBboxPatch((0.03, y(1.15)), 0.94, 1.05 / H,
                 boxstyle="round,pad=0.002,rounding_size=0.006", fc=COR_HEADER, ec="none", zorder=2))
    ax.text(0.06, y(0.52), f"☀  Painel de Energia Injetada — {nome}",
            fontsize=20, fontweight="bold", color="white", va="center", zorder=3)
    ax.text(0.06, y(0.88), f"{EMPRESA}  ·  {UF}", fontsize=11, color="#cfe3e3", va="center", zorder=3)
    ax.text(0.945, y(0.50), f"{MESES_BR[mes.month]}/{mes.year}".upper(), fontsize=17,
            fontweight="bold", color="white", ha="right", va="center", zorder=3)
    ax.text(0.945, y(0.85), "Mês de referência", fontsize=9, color="#cfe3e3", ha="right",
            va="center", zorder=3)

    # ----- KPIs -----
    tot_inj = sum(d["injetada"] for d in ucs)
    tot_cons = sum(d["consumo"] for d in ucs)
    tot_saldo = sum(d["saldo"] for d in ucs)
    cons_med = sum(d["consumo_medio"] for d in ucs)
    autonomia = tot_saldo / cons_med if cons_med else None
    kpis = [
        ("Geração no mês (injetada)", f"{_fmt(tot_inj)} kWh", COR_GERACAO),
        ("Consumo no mês",            f"{_fmt(tot_cons)} kWh", COR_CONSUMO),
        ("Saldo total",               f"{_fmt(tot_saldo)} kWh", COR_HEADER2),
        ("Autonomia (meses de consumo)", "—" if autonomia is None else f"{autonomia:.0f} meses",
         classificar(tot_saldo, autonomia)[1]),
        ("Saldo nos últimos 12 meses", f"{_fmt_sinal(var)} kWh/mês", "#1a7f5a" if var >= 0 else "#b23b3b"),
    ]
    kx0, kw, kgap, ky_top, kh = 0.03, 0.182, 0.0075, 1.45, 1.0
    for i, (lab, val, cor) in enumerate(kpis):
        x = kx0 + i * (kw + kgap)
        ax.add_patch(FancyBboxPatch((x, y(ky_top + kh)), kw, kh / H,
                     boxstyle="round,pad=0.002,rounding_size=0.008",
                     fc="white", ec="#d9e2e4", lw=1, zorder=2))
        ax.add_patch(plt.Rectangle((x, y(ky_top + kh)), 0.006, kh / H, fc=cor, ec="none", zorder=3))
        ax.text(x + 0.02, y(ky_top + 0.30), val, fontsize=15.5, fontweight="bold", color=COR_TXT,
                va="center", zorder=3)
        ax.text(x + 0.02, y(ky_top + 0.72), lab, fontsize=9, color=COR_MUTED, va="center", zorder=3)

    # ----- gráficos: geração x consumo | saldo -----
    def eixo(x0, largura):
        return fig.add_axes([x0, y(y_graf + h_graf), largura, (h_graf - 0.45) / H])
    meses_x = list(range(len(serie)))
    rotulos = [_mes_curto(t) for t in serie.index]

    a1 = eixo(0.065, 0.53)
    _estilo_eixo(a1)
    w = 0.38
    a1.bar([i - w / 2 - 0.01 for i in meses_x], serie["energia_injetada"], w, color=COR_GERACAO,
           label="Geração (energia injetada)", zorder=2)
    a1.bar([i + w / 2 + 0.01 for i in meses_x], serie["consumo"], w, color=COR_CONSUMO,
           label="Consumo das UCs", zorder=2)
    a1.set_xticks(meses_x, rotulos)
    a1.set_xlim(-0.6, len(meses_x) - 0.4)
    a1.legend(loc="upper left", bbox_to_anchor=(0, 1.13), ncol=2, frameon=False, fontsize=8,
              handlelength=1, handleheight=0.8, labelcolor=COR_TXT)
    ax.text(0.065, y(y_graf + 0.02), "Geração × consumo por mês (kWh)", fontsize=10.5,
            fontweight="bold", color=COR_TXT, va="top")

    a2 = eixo(0.665, 0.3)
    _estilo_eixo(a2)
    s = serie["saldo_final"]
    a2.fill_between(meses_x, s, color=COR_SALDO, alpha=0.10, lw=0, zorder=1)
    a2.plot(meses_x, s, color=COR_SALDO, lw=2, marker="o", ms=4, zorder=2)
    a2.set_xticks(meses_x[::2], rotulos[::2])
    a2.set_xlim(-0.4, len(meses_x) - 0.6)
    a2.set_ylim(0, s.max() * 1.18 if s.max() > 0 else 1)
    a2.annotate(_fmt(s.iloc[-1]), (meses_x[-1], s.iloc[-1]), xytext=(0, 7),
                textcoords="offset points", ha="center", fontsize=8, fontweight="bold", color=COR_TXT)
    ax.text(0.665, y(y_graf + 0.02), "Saldo total de créditos (kWh)", fontsize=10.5,
            fontweight="bold", color=COR_TXT, va="top")

    # ----- tabela -----
    cols = [("Unidade", 0.045, "left"), ("Seções", 0.235, "left"),
            ("Consumo\n(kWh)", 0.43, "right"), ("Injetada\n(kWh)", 0.495, "right"),
            ("Recebido\n(kWh)", 0.56, "right"), ("Utilizado\n(kWh)", 0.625, "right"),
            ("Saldo\n(kWh)", 0.69, "right"), ("Autonomia\n(meses)", 0.755, "right"),
            ("% Atual", 0.805, "right"), ("% Ideal", 0.855, "right"),
            ("Status do Saldo", 0.878, "left")]
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.5)), 0.94, 0.5 / H, fc=COR_HEADER2, ec="none", zorder=2))
    for nome_c, cx, al in cols:
        ax.text(cx, y(yh + 0.25), nome_c, fontsize=8.4, fontweight="bold", color="white",
                ha=al, va="center", zorder=3, linespacing=0.9)
    y0 = yh + 0.5
    for i, d in enumerate(ucs):
        yy = y0 + i * row_h
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H,
                     fc=COR_LINHA_A if i % 2 == 0 else COR_LINHA_B, ec="none", zorder=1))
        yc = y(yy + row_h / 2)
        kw_ = dict(fontsize=8.6, va="center", color=COR_TXT, zorder=3)
        ax.text(0.045, yc, d["unidade"], ha="left", fontweight="bold" if d["eh_geradora"] else None, **kw_)
        ax.text(0.235, yc, d["secoes"], ha="left", **{**kw_, "color": COR_MUTED, "fontsize": 8})
        for cx, campo in [(0.43, "consumo"), (0.495, "injetada"), (0.56, "recebido"),
                          (0.625, "utilizado"), (0.69, "saldo")]:
            ax.text(cx, yc, _fmt(d[campo]), ha="right", **kw_)
        ax.text(0.755, yc, "—" if d["autonomia"] is None else f'{d["autonomia"]:.1f}', ha="right", **kw_)
        ax.text(0.805, yc, _fmt_pct(d["pct_atual"]), ha="right", **kw_)
        if d["pct_ideal"] is None:
            ax.text(0.855, yc, "—", ha="right", **{**kw_, "color": "#9aa7a7"})
        else:
            delta = d["pct_ideal"] - d["pct_atual"]
            seta = "▲" if delta > 0.05 else ("▼" if delta < -0.05 else "=")
            cseta = "#1a7f5a" if delta > 0.05 else ("#b23b3b" if delta < -0.05 else COR_MUTED)
            ax.text(0.855, yc, _fmt_pct(d["pct_ideal"]), ha="right", fontweight="bold", **kw_)
            ax.text(0.8595, yc, seta, fontsize=6.6, va="center", ha="left", color=cseta, zorder=3)
        ax.add_patch(FancyBboxPatch((0.878, y(yy + row_h - 0.055)), 0.088, (row_h - 0.11) / H,
                     boxstyle="round,pad=0.001,rounding_size=0.010", fc=d["cor"], ec="none", zorder=3))
        ax.text(0.922, yc, d["status"], fontsize=7.1, fontweight="bold", color="white",
                va="center", ha="center", zorder=4)

    # ----- rodapé esquerdo: rateio da geradora -----
    ax.add_patch(FancyBboxPatch((0.03, y(fy + box_h)), 0.455, box_h / H,
                 boxstyle="round,pad=0.003,rounding_size=0.008", fc="#eaf2fb", ec="#b5cde8", lw=1, zorder=2))
    ax.text(0.05, y(fy + 0.32), titulo, fontsize=11, fontweight="bold", color="#1f4f86", va="center", zorder=3)
    ax.text(0.05, y(fy + 0.62), "\n".join(linhas), fontsize=8.6, color="#1f3b5c", va="top",
            zorder=3, linespacing=1.55)
    ax.text(0.05, y(fy + box_h - 0.14), nota, fontsize=7.4, color="#1f3b5c", style="italic",
            va="bottom", zorder=3, linespacing=1.3)

    # ----- rodapé direito: créditos a expirar -----
    ax.add_patch(FancyBboxPatch((0.515, y(fy + box_h)), 0.455, box_h / H,
                 boxstyle="round,pad=0.003,rounding_size=0.008", fc="#fdf5e6", ec="#e8cf9c", lw=1, zorder=2))
    ax.text(0.535, y(fy + 0.32), "⚠  Créditos a expirar", fontsize=11, fontweight="bold",
            color="#8a5a00", va="center", zorder=3)
    exp_ord = exp.sort_values("prox_expirar_data")
    linhas_e = []
    for r in exp_ord.itertuples():
        origem = "própria" if r.tipo == "Geradora" else f"de {nome_uc(int(r.geradora_uc_conces))}"
        data = r.prox_expirar_data.strftime("%m/%Y") if pd.notna(r.prox_expirar_data) else "—"
        linhas_e.append(f"{r.unidade} ({origem}):  {_fmt(r.prox_expirar_kwh)} kWh em {data}")
    ax.text(0.535, y(fy + 0.62), "\n".join(linhas_e) or "—", fontsize=8.6, color="#5c3d00",
            va="top", zorder=3, linespacing=1.55)
    expirados = exp["expirados_kwh"].sum()
    ax.text(0.535, y(fy + box_h - 0.14),
            f"Próxima parcela do saldo a expirar, por seção. Expirados no último ciclo: {_fmt(expirados)} kWh.",
            fontsize=7.4, color="#5c3d00", style="italic", va="bottom", zorder=3)

    # ----- notas -----
    hoje = dt.date.today().strftime("%d/%m/%Y")
    ax.text(0.03, y(fy + box_h + 0.3),
            f"Gerado em {hoje}  ·  Fonte: {', '.join(sorted(set(exp['arquivo'])))}  ·  "
            f"Status pela autonomia (saldo ÷ consumo médio de {JANELA_MEDIA_MESES} meses): "
            f"Superávit > {LIM_MESES_SUPERAVIT} · Confortável > {LIM_MESES_CONFORTAVEL} · "
            f"Crítico/Zero < {LIM_BAIXO} kWh  (parâmetros configuráveis)",
            fontsize=7, color="#8a9797", va="center")
    ax.text(0.03, y(fy + box_h + 0.55),
            "Saldo de cada UC = soma de todas as origens (geração própria + créditos recebidos de outra "
            "geradora). Rateio de cada geradora no relatório PDF.",
            fontsize=7, color="#8a9797", va="center")

    saida_base = Path(saida_base)
    png, pdf = saida_base.with_suffix(".png"), saida_base.with_suffix(".pdf")
    fig.savefig(png, facecolor=COR_FUNDO, bbox_inches="tight", pad_inches=0.15)
    fig.savefig(pdf, facecolor=COR_FUNDO, bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)
    logger.info("Painel salvo: %s e %s", png.name, pdf.name)
    return png, pdf


# ====================================================================== 5.3 PAINEL PADRÃO PR
COR_KPI = "#ffffff"
RESUMO_RATEIO: dict = {}


def classificar_pr(saldo: float):
    """(rótulo, cor) pelo saldo em kWh — mesmos limiares do painel PR."""
    if saldo > LIM_SUPERAVIT:
        return "Superávit", "#1a7f5a"
    if saldo > LIM_CONFORTAVEL:
        return "Confortável", "#2f7fb0"
    if saldo >= LIM_BAIXO:
        return "Saldo baixo", "#c98a1a"
    return "Crítico / Zero", "#b23b3b"


def _wrap(s: str, width: int) -> str:
    out, line = [], ""
    for w in s.split(", "):
        if len(line) + len(w) + 2 > width:
            out.append(line.rstrip(", ")); line = ""
        line += w + ", "
    out.append(line.rstrip(", "))
    return "\n".join([x for x in out if x])


def dados_painel_pr(base: pd.DataFrame, rateio: dict | None = None, geradora: int | None = None):
    """Uma linha por seção no último mês (saldo NÃO somado entre origens). Devolve (dados, mes).
    `geradora` = UC da geradora do rateio (padrão: a que tem beneficiárias)."""
    global RESUMO_RATEIO
    b = base.copy()
    b["referencia"] = pd.to_datetime(b["referencia"])
    mes = b["referencia"].max()
    if geradora is None:
        gs = geradoras_ordenadas(b)
        geradora = gs[0] if gs else None
    dados = []
    for sec in secoes_relatorio(b):
        g = b[(b["uc_conces"] == sec["uc"]) & (b["geradora_uc_conces"] == sec["origem"])]
        g_mes = g[g["referencia"] == mes]
        if g_mes.empty:
            continue
        col_inj = "energia_injetada" if sec["geradora"] else "cred_receb"
        cons = g.groupby("referencia")["energia_ativa"].sum().sort_index()
        jan = cons.iloc[-int(JANELA_MEDIA_MESES):] if JANELA_MEDIA_MESES else cons
        dados.append(dict(
            loja=sec["rotulo"], unidade=sec["unidade"], uc=sec["uc"], origem=sec["origem"],
            geradora=sec["geradora"],
            benef=(not sec["geradora"]) and sec["origem"] == geradora,     # entra no rateio do painel
            consumo=float(g_mes["energia_ativa"].sum()),
            injetada=float(g_mes[col_inj].sum()),
            media_inj=float(g.groupby("referencia")[col_inj].sum().mean()),
            saldo=float(g_mes["saldo_final"].sum()),
            pct_atual=float(g_mes["percentual"].max()),
            consumo_medio=float(jan.mean()) if len(jan) else 0.0,
        ))

    # % ideal: rateio anual (estratégia W) da geradora; conversão % -> kWh pelo fator do mês
    benef = [d for d in dados if d["benef"]]
    total_pct = sum(d["pct_atual"] for d in benef)
    total_inj = sum(d["injetada"] for d in benef)
    fator = (total_inj / total_pct) if total_pct else 0.0
    tab = rateio["tabela"] if rateio else None
    for d in dados:
        if d["benef"] and tab is not None and d["unidade"] in tab.index:
            d["pct_ideal"] = float(tab.loc[d["unidade"], "pct_ideal"])
            d["inj_ideal_kwh"] = round(d["pct_ideal"] * fator)
            d["delta_kwh"] = round(d["inj_ideal_kwh"] - d["injetada"])
        else:
            d["pct_ideal"] = d["inj_ideal_kwh"] = None
            d["delta_kwh"] = 0
        # status pelo saldo UTILIZÁVEL (saldo preso não compensa a fatura)
        d["preso"] = float(tab["saldo_preso"].get(d["unidade"], 0.0)) if (tab is not None and d["benef"]) else 0.0
        d["status"], d["cor"] = classificar_pr(d["saldo"] - d["preso"])
        if d["preso"] > 0 and d["saldo"] - d["preso"] <= LIM_CONFORTAVEL:
            d["status"], d["cor"] = "Saldo preso", "#6b5b95"

    RESUMO_RATEIO = dict(total_pct=total_pct, fator=fator,
                         geradora=nome_uc(geradora) if geradora else "—")
    if rateio:
        RESUMO_RATEIO.update(k=rateio["k"], cobertura_anual=rateio["cobertura_estrutural"],
                             inicio=rateio["parametros"]["Início do ciclo"])
    return dados, mes


def desenhar_painel_pr(dados: list, maxmes, fontes: str = ""):
    """Painel executivo no mesmo layout do painel PR (Copel)."""
    dados = sorted(dados, key=lambda d: (d["geradora"], d["loja"]))
    n = len(dados)
    row_h = 0.30
    fig_h = 3.2 + n * row_h + 3.45          # cabeçalho+KPIs+tabela, caixas do rodapé e notas
    fig = plt.figure(figsize=(13.6, fig_h), dpi=140)
    fig.patch.set_facecolor(COR_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    H = fig_h
    def y(px): return 1 - px / H

    # ----- cabeçalho -----
    ax.add_patch(FancyBboxPatch((0.03, y(1.15)), 0.94, 1.05 / H, boxstyle="round,pad=0.002,rounding_size=0.006",
                 fc=COR_HEADER, ec="none", transform=ax.transAxes, zorder=2))
    ax.text(0.06, y(0.52), "\u2600  Painel de Energia Injetada \u2014 Geração Distribuída",
            fontsize=20, fontweight="bold", color="white", va="center", zorder=3)
    ax.text(0.06, y(0.88), f"{EMPRESA}  \u00b7  {UF}", fontsize=11, color="#cfe3e3", va="center", zorder=3)
    ax.text(0.945, y(0.50), f"{MESES_BR[maxmes.month]}/{maxmes.year}".upper(), fontsize=17,
            fontweight="bold", color="white", ha="right", va="center", zorder=3)
    ax.text(0.945, y(0.85), "Mês de referência", fontsize=9, color="#cfe3e3", ha="right", va="center", zorder=3)

    # ----- KPIs (só beneficiárias da geradora do rateio) -----
    benef = [d for d in dados if d["benef"]]
    tot_cons = sum(d["consumo"] for d in benef)
    tot_inj = sum(d["injetada"] for d in benef)
    cob = (tot_inj / tot_cons * 100) if tot_cons else 0
    kpis = [("Consumo total", f"{_fmt(tot_cons)} kWh", COR_HEADER2),
            ("Energia injetada", f"{_fmt(tot_inj)} kWh", "#1a7f5a"),
            ("Cobertura média", f"{cob:.0f}%", "#2f7fb0"),
            ("UCs c/ superávit", str(sum(d["status"] == "Superávit" for d in benef)), "#1a7f5a"),
            ("UCs críticas", str(sum(d["status"] == "Crítico / Zero" for d in benef)), "#b23b3b")]
    kx0, kw, kgap = 0.03, 0.182, 0.0075
    ky_top, kh = 1.45, 1.0
    for i, (lab, val, cor) in enumerate(kpis):
        x = kx0 + i * (kw + kgap)
        ax.add_patch(FancyBboxPatch((x, y(ky_top + kh)), kw, kh / H, boxstyle="round,pad=0.002,rounding_size=0.008",
                     fc=COR_KPI, ec="#d9e2e4", lw=1, transform=ax.transAxes, zorder=2))
        ax.add_patch(plt.Rectangle((x, y(ky_top + kh)), 0.006, kh / H, fc=cor, ec="none",
                     transform=ax.transAxes, zorder=3))
        ax.text(x + 0.02, y(ky_top + 0.30), val, fontsize=15.5, fontweight="bold", color=cor, va="center", zorder=3)
        ax.text(x + 0.02, y(ky_top + 0.72), lab, fontsize=9, color="#5a6b6b", va="center", zorder=3)

    # ----- tabela -----
    cols = [("Unidade", 0.045, "left"), ("Consumida\n(kWh)", 0.40, "right"),
            ("Injetada\n(kWh)", 0.475, "right"), ("Média Inj.\n(~kWh)", 0.55, "right"),
            ("Saldo\n(kWh)", 0.625, "right"), ("% Atual", 0.685, "right"), ("% Ideal", 0.745, "right"),
            ("Inj. p/ Ideal\n(~kWh)", 0.845, "right"), ("Status do Saldo", 0.868, "left")]
    yh = 2.7
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.5)), 0.94, 0.5 / H, fc=COR_HEADER2, ec="none",
                 transform=ax.transAxes, zorder=2))
    for name, cx, al in cols:
        ax.text(cx, y(yh + 0.25), name, fontsize=8.4, fontweight="bold", color="white",
                ha=al, va="center", zorder=3, linespacing=0.9)
    y0 = yh + 0.5
    for i, d in enumerate(dados):
        yy = y0 + i * row_h
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H,
                     fc=COR_LINHA_A if i % 2 == 0 else COR_LINHA_B, ec="none", transform=ax.transAxes, zorder=1))
        yc = y(yy + row_h / 2)
        ax.text(0.045, yc, d["loja"], fontsize=8.6, color=COR_TXT, va="center", ha="left", zorder=3)
        ax.text(0.40, yc, _fmt(d["consumo"]), fontsize=8.6, va="center", ha="right", zorder=3)
        ax.text(0.475, yc, _fmt(d["injetada"]), fontsize=8.6, va="center", ha="right", zorder=3)
        ax.text(0.55, yc, _fmt(d["media_inj"]), fontsize=8.6, va="center", ha="right", color="#5a6b6b", zorder=3)
        ax.text(0.625, yc, _fmt(d["saldo"]), fontsize=8.6, va="center", ha="right", zorder=3)
        ax.text(0.685, yc, f'{d["pct_atual"]:.2f}', fontsize=8.6, va="center", ha="right", zorder=3)
        if d["pct_ideal"] is None:
            ax.text(0.745, yc, "\u2014", fontsize=8.6, va="center", ha="right", color="#9aa7a7", zorder=3)
            ax.text(0.828, yc, "\u2014", fontsize=8.6, va="center", ha="right", color="#9aa7a7", zorder=3)
        else:
            delta = d["pct_ideal"] - d["pct_atual"]
            seta = "\u25b2" if delta > 0.05 else ("\u25bc" if delta < -0.05 else "=")
            cseta = "#1a7f5a" if delta > 0.05 else ("#b23b3b" if delta < -0.05 else "#5a6b6b")
            ax.text(0.745, yc, f'{d["pct_ideal"]:.2f}', fontsize=8.6, fontweight="bold", va="center", ha="right", zorder=3)
            ax.text(0.751, yc, seta, fontsize=6.6, va="center", ha="left", color=cseta, zorder=3)
            ax.text(0.828, yc, _fmt(d["inj_ideal_kwh"]), fontsize=8.6, fontweight="bold", va="center", ha="right", zorder=3)
            dk = d["delta_kwh"]
            cdk = "#1a7f5a" if dk > 0 else ("#b23b3b" if dk < 0 else "#5a6b6b")
            sinal = "+" if dk > 0 else ("\u2212" if dk < 0 else "\u00b1")
            ax.text(0.862, yc, f"{sinal}{_fmt(abs(dk))}", fontsize=6.4, va="center", ha="right", color=cdk, zorder=3)
        ax.add_patch(FancyBboxPatch((0.868, y(yy + row_h - 0.055)), 0.098, (row_h - 0.11) / H,
                     boxstyle="round,pad=0.001,rounding_size=0.010", fc=d["cor"], ec="none",
                     transform=ax.transAxes, zorder=3))
        ax.text(0.917, yc, d["status"], fontsize=7.1, fontweight="bold", color="white", va="center", ha="center", zorder=4)

    # ----- rodapé: recomendações -----
    fy = y0 + n * row_h + 0.35
    criticas = sorted([d for d in benef if d["status"] == "Crítico / Zero"], key=lambda d: d["consumo"], reverse=True)
    superavit = sorted([d for d in benef if d["status"] == "Superávit"], key=lambda d: d["saldo"], reverse=True)
    box_h = 2.15
    ax.add_patch(FancyBboxPatch((0.03, y(fy + box_h)), 0.455, box_h / H, boxstyle="round,pad=0.003,rounding_size=0.008",
                 fc="#fdecec", ec="#e6b3b3", lw=1, transform=ax.transAxes, zorder=2))
    ax.text(0.05, y(fy + 0.32), "\u26a0  Atenção urgente \u2014 reforçar alocação",
            fontsize=11, fontweight="bold", color="#b23b3b", va="center", zorder=3)
    nomes_c = ", ".join(f'{d["loja"]} ({_fmt_sinal(d["delta_kwh"])})' for d in criticas[:10])
    ax.text(0.05, y(fy + 0.78), _wrap(nomes_c, 60) or "\u2014", fontsize=8.4, color="#7a2e2e",
            va="top", zorder=3, linespacing=1.35)
    ganho_tot = sum(d["delta_kwh"] for d in criticas)
    ax.text(0.05, y(fy + box_h - 0.18),
            f"{len(criticas)} UC(s) com saldo zerado/negativo.\n"
            f"Variação total com a % Ideal: {_fmt_sinal(ganho_tot)} kWh/mês (valor por UC entre parênteses).",
            fontsize=7.4, color="#7a2e2e", style="italic", va="bottom", zorder=3, linespacing=1.3)
    ax.add_patch(FancyBboxPatch((0.515, y(fy + box_h)), 0.455, box_h / H, boxstyle="round,pad=0.003,rounding_size=0.008",
                 fc="#e9f6ef", ec="#a9d9c1", lw=1, transform=ax.transAxes, zorder=2))
    ax.text(0.535, y(fy + 0.32), "\u2714  Superávit acumulado \u2014 reduzir alocação",
            fontsize=11, fontweight="bold", color="#1a7f5a", va="center", zorder=3)
    nomes_s = ", ".join(f'{d["loja"]} ({_fmt_sinal(d["delta_kwh"])})' for d in superavit[:8])
    ax.text(0.535, y(fy + 0.78), _wrap(nomes_s, 58) or "\u2014", fontsize=8.4, color="#1c5c44",
            va="top", zorder=3, linespacing=1.35)
    ax.text(0.535, y(fy + box_h - 0.18),
            "Créditos podem expirar (ReN ANEEL).\nReduzir a % destas UCs (% Ideal \u25bc) "
            "para consumir a sobra até a próxima revisão.",
            fontsize=7.4, color="#1c5c44", style="italic", va="bottom", zorder=3, linespacing=1.3)

    # ----- notas -----
    hoje = dt.date.today().strftime("%d/%m/%Y")
    ax.text(0.03, y(fy + box_h + 0.35),
            f"Gerado automaticamente em {hoje}  \u00b7  Fonte: {fontes}  \u00b7  "
            f"Regras: Superávit > {_fmt(LIM_SUPERAVIT)} kWh \u00b7 Confortável > {_fmt(LIM_CONFORTAVEL)} \u00b7 "
            f"Baixo \u2265 {LIM_BAIXO} \u00b7 Crítico/Zero < {LIM_BAIXO}  (parâmetros configuráveis)",
            fontsize=7, color="#8a9797", va="center", zorder=3)
    rr = RESUMO_RATEIO or {}
    nota = (f"Uma linha por seção do demonstrativo (UC \u00d7 geradora de origem); saldo não somado entre origens. "
            f"KPIs e % Ideal: beneficiárias da geradora {rr.get('geradora', '—')}")
    if rr.get("k") is not None:
        nota += (f" \u2014 estratégia W (k = {rr['k'] * 100:.1f}%, ciclo a partir de {rr['inicio']}); "
                 f"\u03a3% = {rr['total_pct']:.2f}; ~{rr['fator']:.0f} kWh por 1% no mês.")
    ax.text(0.03, y(fy + box_h + 0.62), nota, fontsize=7, color="#8a9797", va="center", zorder=3)
    return fig


def gerar_painel_pr(dados: list, maxmes, saida_base: Path, fontes: str = ""):
    fig = desenhar_painel_pr(dados, maxmes, fontes)
    saida_base = Path(saida_base)
    png, pdf = saida_base.with_suffix(".png"), saida_base.with_suffix(".pdf")
    fig.savefig(png, facecolor=COR_FUNDO, bbox_inches="tight", pad_inches=0.15)
    fig.savefig(pdf, facecolor=COR_FUNDO, bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)
    logger.info("Painel (padrão PR) salvo: %s e %s", png.name, pdf.name)
    return png, pdf


## Etapa 4 — Rateio anual (% fixo por 12 meses)

O % de cada loja só pode ser alterado **uma vez por ano**, então o objetivo não é o % "ótimo hoje", e sim
um % que **aguente os 12 meses** sem deixar saldo parado numa loja enquanto outra paga a fatura.
Este bloco é **o mesmo nos notebooks Copel, CELESC e Nelson**; só o *adaptador* (4.1) muda, porque cada
planilha guarda o `Demonstrativo` num formato diferente.

### Modelo

Com o % fixo `pᵢ` e a energia distribuída no mês `Gₜ`, o saldo de cada loja evolui assim:

```
Sᵢ,ₜ = Sᵢ,ₜ₋₁ + pᵢ·Gₜ − Compᵢ,ₜ        Compᵢ,ₜ = min(Sᵢ,ₜ₋₁ + pᵢ·Gₜ ; Uᵢ,ₜ)
Uᵢ,ₜ = max(0 ; Consumoᵢ,ₜ − CustoDispᵢ − GeraçãoPrópriaᵢ,ₜ)   ← consumo COMPENSÁVEL
Faturadoᵢ,ₜ = Uᵢ,ₜ − Compᵢ,ₜ
```

**Regra de rateio — cobertura equalizada ("water-filling"):**

```
aᵢ = max(0 ; k·Uᵢ,ano + Alvoᵢ − Sᵢ,início)      com k tal que  Σ aᵢ = G_ano
pᵢ = aᵢ / Σ aᵢ   (× Σ% atual, arredondado conservando a soma)
```

`k` é a **cobertura comum**: a maior cobertura que dá para garantir a todas as lojas ao mesmo tempo.
Lojas cujo saldo já passa de `k·U` recebem 0% e consomem o saldo; as demais ficam com a mesma cobertura.

### Tratamentos

| Tema | Como o bloco trata |
|---|---|
| Custo de disponibilidade | informado pelo adaptador (PDF), inferido dos dados (consumo − crédito usado) ou `CUSTO_DISP_PADRAO`; `CUSTO_DISP_MANUAL` sobrepõe tudo |
| Energia distribuível `G` | informada pelo adaptador (ex.: excedentes distribuídos) ou Σ créditos recebidos reescalada para 100% × `FATOR_GERACAO` |
| Mês incompleto | se as beneficiárias ainda não receberam o crédito do último mês, a análise para no último mês completo |
| Previsão de consumo | perfil dos últimos 12 meses × tendência amortecida (`TENDENCIA_*`) |
| Geração própria | UC que também é geradora só precisa do consumo que a própria geração não cobre |
| Saldo preso | linha de cadastro sem consumo com saldo, ou UC cujo saldo não é usado há `MESES_SALDO_PRESO` meses → não conta como saldo |
| Sazonalidade | a simulação começa em `MES_INICIO_CICLO`; o saldo-alvo inclui o **colchão sazonal** exigido por esse mês |
| Meses até a troca | simulados com o **% atual** para estimar o saldo no início do ciclo |
| Sobra estrutural | se geração + saldo passam do consumo, o relatório indica a parcela que poderia ser cedida a outras UCs |

### Validação

- **Comparação de métodos** (atual, legado 12,5×, proporcional, necessidade − saldo, cobertura equalizada,
  otimização numérica) com o % **fixo** sob os `CENARIOS` de estresse.
- **Backtest com origem móvel**: roda automaticamente quando houver **≥ 24 meses** de histórico.

### 4.0 Parâmetros do rateio anual

In [ ]:
# ---- Rateio anual (Etapas 4 e 5) — mesmos parâmetros nos notebooks Copel, CELESC e Nelson
ESTRATEGIA_RATEIO  = "W"      # "W" = cobertura anual equalizada (water-filling) — RECOMENDADO
                              # "C" / "A" = regras legadas da v1 (a % Ideal do painel volta a usá-las)
MES_INICIO_CICLO   = 10       # mês (1-12) em que o novo % passa a valer no site da distribuidora
FATOR_GERACAO      = 1.00     # geração do próximo ciclo ÷ últimos 12 meses (ex.: 0.995 = degradação dos módulos)
TENDENCIA_AMORTECIMENTO = 0.5 # 0 = ignora tendência de consumo; 1 = aplica a tendência inteira
TENDENCIA_LIMITES  = (0.85, 1.25)   # limites do fator de tendência por loja
USAR_BUFFER_SAZONAL = True    # saldo-alvo inclui o colchão sazonal exigido pelo mês de início do ciclo
MARGEM_SEGURANCA   = 0.03     # + fração do consumo anual compensável no saldo-alvo (erro de previsão)
CUSTO_DISP_PADRAO  = 100      # kWh/mês não compensável quando não há outra fonte (TR=100, BI=50, MO=30)
CUSTO_DISP_MANUAL: dict = {}  # {uc: kWh} — sobrepõe PDF / inferência
MESES_SALDO_PRESO  = 3        # nº de meses sem uso do saldo (com consumo) para classificá-lo como "preso"
PESO_SALDO_PRESO   = 0.5      # fração do saldo preso contada como saldo no rateio (só lojas ativas):
                              #   0 = o saldo nunca será liberado  -> a loja recebe % para cobrir tudo sozinha
                              #   1 = a distribuidora vai realocar o saldo antes do ciclo -> a loja recebe menos %
                              #   0.5 = meio-termo: limita o erro nos dois casos (falta vs. sobra de crédito)
AUTO_EXCLUIR_SEM_COMPENSACAO = True  # loja cujo saldo não é usado há MESES_SALDO_PRESO meses recebe 0%
EXCLUIR_DO_RATEIO: list = []  # nomes de loja forçados a 0% (ex.: loja que vai fechar)
RODAR_OTIMIZACAO   = True     # inclui o método "Otimização numérica" na comparação (mais lento)
CENARIOS = {                  # (fator geração, fator consumo) — o % é mantido FIXO em todos
    "Base":                        (1.00, 1.00),
    "Geração −15%":                (0.85, 1.00),
    "Geração +10%":                (1.10, 1.00),
    "Consumo +10%":                (1.00, 1.10),
    "Geração −15% e Consumo +10%": (0.85, 1.10),
}
ABAS_RATEIO = ["Rateio Anual", "Rateio Simulação", "Rateio Métodos", "Rateio Premissas"]
GERAR_ABAS_RATEIO = False     # False = o .xlsx não recebe as abas "Rateio *" (o rateio fica só no relatório PDF)

### 4.1 Adaptador Nelson — `entrada_rateio_nelson()`

Com o relatório único, o adaptador roda **uma vez por geradora** da base consolidada (`geradora=`):

- **Com beneficiárias** (ex.: Nelson Souza → Butiatuvinha 68% e Chácara 32%): a energia distribuível é o
  *Saldo Transf. Outra UC* da geradora; o custo de disponibilidade é inferido dos dados. UCs que também
  são geradoras (a Chácara) só precisam do consumo que a própria geração não cobre.
- **Sem beneficiárias** (ex.: Chácara Nelson, 100% para si mesma): a própria UC é a única do rateio; o
  relatório mostra a sobra estrutural e quanto da geração poderia ser cedida a outras UCs.

In [ ]:
def _num_s(s):
    return pd.to_numeric(s, errors="coerce").fillna(0.0)


def entrada_rateio_nelson(nome: str, base: pd.DataFrame, propria: dict | None = None,
                          geradora: int | None = None) -> dict:
    """Converte a base consolidada de um demonstrativo para o formato padrão do bloco de rateio.
    `propria` = {uc: série mensal de energia injetada} das UCs que são geradoras (em qualquer demonstrativo)."""
    b = base.copy()
    b["referencia"] = pd.to_datetime(b["referencia"])
    ger = b.loc[b["tipo"] == "Geradora", "uc_conces"]
    if ger.empty:
        raise RuntimeError(f"{nome}: nenhuma seção Geradora no demonstrativo.")
    geradora = int(geradora) if geradora is not None else int(ger.iloc[0])
    benef = b[(b["tipo"] == "Beneficiária") & (b["geradora_uc_conces"] == geradora)]

    def _std(fr, cred_col, pct, geradora_flag, consumo_col="energia_ativa"):
        return pd.DataFrame({
            "loja": fr["unidade"].astype(str), "uc": fr["uc_conces"].astype("int64"),
            "ref": fr["referencia"].dt.to_period("M"),
            "consumo": _num_s(fr[consumo_col]) if consumo_col else 0.0,
            "saldo_ant": _num_s(fr["saldo_anterior"]), "cred": _num_s(fr[cred_col]),
            "pct": pct if not isinstance(pct, str) else _num_s(fr[pct]),
            "saldo_fin": _num_s(fr["saldo_final"]), "geradora": geradora_flag,
        })

    if benef.empty:
        # a geradora só abastece a si mesma: ela é a única UC do próprio rateio (100%)
        g_rows = b[(b["uc_conces"] == geradora) & (b["tipo"] == "Geradora")]
        outras = b[(b["uc_conces"] == geradora) & (b["tipo"] != "Geradora")]   # créditos vindos de outra geradora
        partes = [_std(g_rows, "energia_injetada", "percentual", False)]
        if not outras.empty:
            partes.append(_std(outras, "cred_receb", 0.0, False, consumo_col=None))
        std = pd.concat(partes, ignore_index=True)
        G = _num_s(g_rows["energia_injetada"]).groupby(g_rows["referencia"].dt.to_period("M")).sum()
        proprio = {}
        sistema = f"Geradora {nome} (geração própria, sem beneficiárias)"
    else:
        g_rows = b[(b["tipo"] == "Geradora") & (b["uc_conces"] == geradora)]
        std = pd.concat([_std(benef, "cred_receb", "percentual", False),
                         _std(g_rows, "cred_receb", 0.0, True)], ignore_index=True)
        G = _num_s(g_rows["saldo_transferido"]).groupby(g_rows["referencia"].dt.to_period("M")).sum()
        ucs_benef = set(benef["uc_conces"].astype("int64"))
        proprio = {uc: s for uc, s in (propria or {}).items() if uc in ucs_benef and uc != geradora}
        sistema = f"Geradora {nome} (UC {geradora:015d})"
    return dict(df=std, G=G, inferir_cd=True, geracao_propria=proprio, sistema=sistema,
                fonte=", ".join(sorted(set(b["arquivo"].dropna().astype(str)))), portal="site da Copel")

### 4.2 Base de dados do rateio (formato padrão, vindo do adaptador)

O adaptador entrega um `DataFrame` com **uma linha por UC × seção × mês** e as colunas
`loja, uc, ref, consumo, saldo_ant, cred, pct, saldo_fin, geradora` (+ opcionais: `G` = energia
distribuível por mês, `cd` = custo de disponibilidade por UC, `geracao_propria`, `novas`, `alertas`).

In [ ]:
import io as _io
import numpy as np
import datetime as dt
from pathlib import Path
try:
    from scipy.optimize import minimize
except Exception:                                   # noqa: BLE001
    minimize = None

MESES_ABREV = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]


def _rotulo_mes(m: int) -> str:
    return MESES_ABREV[(m - 1) % 12]


def _r_fmt(n) -> str:
    try:
        return f"{float(n):,.0f}".replace(",", ".")
    except (TypeError, ValueError):
        return str(n)


def _r_fmt_sinal(n) -> str:
    n = round(float(n))
    return ("+" if n > 0 else ("−" if n < 0 else "±")) + _r_fmt(abs(n))


def _r_arred(valores: dict, casas: int = 2) -> dict:
    """Arredonda mantendo a soma (maior resto) — a soma digitada no site fecha exata."""
    q = 10 ** casas
    alvo = round(sum(valores.values()) * q)
    base = {k: int(v * q) for k, v in valores.items()}
    falta = alvo - sum(base.values())
    for k in sorted(valores, key=lambda k: valores[k] * q - base[k], reverse=True)[:max(falta, 0)]:
        base[k] += 1
    return {k: v / q for k, v in base.items()}


def _inferir_cd(por_uc: pd.DataFrame) -> dict:
    """Custo de disponibilidade pelos dados: mediana de (consumo − crédito usado) nos meses em que sobrou
    saldo (o consumo foi compensado até o limite), aproximada para 0/30/50/100 kWh."""
    ok = por_uc[(por_uc["saldo"] > 0) & (por_uc["consumo"] > 0) & (por_uc["usado"] > 0)]
    res = {}
    for uc, g in ok.groupby("uc"):
        v = float((g["consumo"] - g["usado"]).clip(lower=0).median())
        res[uc] = float(min([0, 30, 50, 100], key=lambda x: abs(x - v)))
    return res


def montar_base_rateio(entrada: dict) -> dict:
    """Séries mensais por loja a partir do formato padrão do adaptador."""
    df = entrada["df"].copy()
    df["ref"] = pd.PeriodIndex(df["ref"], freq="M")
    df["uc"] = df["uc"].astype("int64")
    for c in ["consumo", "saldo_ant", "cred", "pct", "saldo_fin"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    ger = df["geradora"].astype(bool)
    b = df[~ger].copy()
    meses_all = pd.period_range(b["ref"].min(), b["ref"].max(), freq="M")
    alertas = list(entrada.get("alertas") or [])

    # ---- energia distribuível e último mês COMPLETO (crédito já lançado nas beneficiárias)
    G_dist = b.groupby("ref")["cred"].sum().reindex(meses_all).fillna(0.0)
    G_ext = entrada.get("G")
    ult = meses_all[-1]
    if G_ext is not None:
        G_ext = pd.Series(G_ext).copy()
        G_ext.index = pd.PeriodIndex(G_ext.index, freq="M")
        ult_g = G_ext.index.max()                               # último mês com dado da geradora
        G_ext = G_ext.groupby(level=0).sum().reindex(meses_all).fillna(0.0)
        # mês além do último da geradora (só parte das beneficiárias já tem o mês) não conta como completo
        completos = [m for m in meses_all
                     if m <= ult_g and (G_ext[m] <= 0 or G_dist[m] >= 0.5 * G_ext[m])]
        ult = max(completos) if completos else ult
        if ult < meses_all[-1]:
            alertas.append(dict(tipo="Mês incompleto", loja="—", kwh=float(G_ext[meses_all[-1]]),
                                detalhe=f"de {ult + 1} a {meses_all[-1]} o crédito da geradora ainda não aparece "
                                        f"em todas as beneficiárias — a análise usa até {ult}"))
    meses = pd.period_range(meses_all[0], ult, freq="M")
    b = b[b["ref"] <= ult]
    df = df[df["ref"] <= ult]

    if "usado" in b.columns and b["usado"].notna().any():          # informado pelo adaptador (ex.: Celesc)
        b["usado"] = pd.to_numeric(b["usado"], errors="coerce").fillna(0.0).clip(lower=0)
    else:                                                          # balanço: anterior + recebido − final
        b["usado"] = (b["saldo_ant"] + b["cred"] - b["saldo_fin"]).clip(lower=0)
    por_uc = (b.groupby(["loja", "uc", "ref"], as_index=False)
                .agg(consumo=("consumo", "sum"), usado=("usado", "sum"),
                     saldo=("saldo_fin", "sum"), cred=("cred", "sum")))

    # ---- custo de disponibilidade: manual > adaptador > inferido > padrão
    cd_ent = dict(entrada.get("cd") or {})
    cd_inf = _inferir_cd(por_uc) if entrada.get("inferir_cd") else {}
    origem_cd = {}
    def _cd(uc):
        for fonte, d in (("manual", CUSTO_DISP_MANUAL), ("PDF", cd_ent), ("inferido", cd_inf)):
            if uc in d and d[uc] is not None and not pd.isna(d[uc]):
                origem_cd[uc] = fonte
                return float(d[uc])
        origem_cd[uc] = "padrão"
        return float(CUSTO_DISP_PADRAO)
    por_uc["cd"] = por_uc["uc"].map(_cd)

    # ---- geração própria (UC que também é geradora em outro demonstrativo)
    proprio = entrada.get("geracao_propria") or {}
    por_uc["proprio"] = 0.0
    for uc, serie in proprio.items():
        s = pd.Series(serie).copy(); s.index = pd.PeriodIndex(s.index, freq="M")
        m = por_uc["uc"] == uc
        por_uc.loc[m, "proprio"] = por_uc.loc[m, "ref"].map(s.groupby(level=0).sum()).fillna(0.0).values
    por_uc["U"] = (por_uc["consumo"] - por_uc["cd"] - por_uc["proprio"]).clip(lower=0)

    def _pivot(frame, col):
        return (frame.pivot_table(index="loja", columns="ref", values=col, aggfunc="sum")
                     .reindex(columns=meses).fillna(0.0))
    C, U, S_mes = _pivot(por_uc, "consumo"), _pivot(por_uc, "U"), _pivot(por_uc, "saldo")

    # ---- energia distribuível a 100% das beneficiárias
    pct_mes = b.groupby(["ref", "uc"])["pct"].max().groupby("ref").sum().reindex(meses)
    if G_ext is not None:
        G = G_ext.reindex(meses).fillna(0.0)
    else:
        G = (G_dist.reindex(meses) * 100 / pct_mes.where(pct_mes > 0)).fillna(G_dist.reindex(meses))

    # ---- saldo preso (a) linha de cadastro sem consumo e com saldo no último mês
    lb = b[b["ref"] == ult]
    preso_a = (lb[(lb["consumo"] <= 0) & (lb["saldo_fin"] > 0) & (lb["cred"] <= 0)]
               .groupby("loja")["saldo_fin"].sum())
    # ---- (b) UC com consumo compensável e saldo, mas sem uso nos últimos N meses
    jan = por_uc[por_uc["ref"].isin(meses[-MESES_SALDO_PRESO:])]
    agg = jan.groupby(["loja", "uc"]).agg(U=("U", "sum"), usado=("usado", "sum"))
    s_ult = por_uc[por_uc["ref"] == ult].set_index(["loja", "uc"])["saldo"].reindex(agg.index).fillna(0.0)
    cond = (agg["U"] > 0) & (agg["usado"] <= 0.01 * agg["U"]) & (s_ult > 0)
    preso_b = s_ult[cond].groupby(level=0).sum()
    sem_comp = sorted(preso_b.index)

    lojas = list(C.index)
    S_total = S_mes[ult].reindex(lojas).fillna(0.0)
    preso = pd.concat([preso_a, preso_b], axis=1).max(axis=1).reindex(lojas).fillna(0.0)
    preso = np.minimum(preso, S_total)
    S_util = (S_total - preso).clip(lower=0)
    pct_atual = (lb.groupby(["loja", "uc"])["pct"].max()          # 1 linha "real" por UC
                   .groupby(level=0).sum().reindex(lojas).fillna(0.0))  # loja com 2+ UCs soma
    ucs = b.groupby("loja")["uc"].unique().map(lambda a: ", ".join(map(str, sorted(a))))

    # ---- alertas
    for l in lojas:
        if preso[l] > 0:
            motivo = (f"saldo não é usado há ≥ {MESES_SALDO_PRESO} meses apesar do consumo"
                      if l in sem_comp else "saldo em cadastro antigo/inativo (sem consumo)")
            alertas.append(dict(tipo="Saldo preso", loja=l, kwh=float(preso[l]),
                                detalhe=motivo + " — solicitar realocação/regularização à distribuidora"))
    for l in sem_comp:
        alertas.append(dict(tipo="Sem compensação", loja=l, kwh=float(C.loc[l].iloc[-MESES_SALDO_PRESO:].sum()),
                            detalhe="consumo não está sendo compensado; % ideal = 0 até regularizar"
                                    if AUTO_EXCLUIR_SEM_COMPENSACAO else "consumo não está sendo compensado"))
    ult12 = C.iloc[:, -12:]
    for l in lojas:
        n_obs = int((ult12.loc[l] > 0).sum())
        if 0 < n_obs < min(12, ult12.shape[1]):
            alertas.append(dict(tipo="Histórico incompleto", loja=l, kwh=None,
                                detalhe=f"só {n_obs} de 12 meses com consumo — meses faltantes estimados pela média"))
    pct_loja_mes = b.groupby(["loja", "ref"])["pct"].max().unstack().reindex(index=lojas, columns=meses).fillna(0)
    cred_loja_mes = b.groupby(["loja", "ref"])["cred"].sum().unstack().reindex(index=lojas, columns=meses).fillna(0)
    for l in lojas:
        falhas = [m for m in meses[-12:] if pct_loja_mes.loc[l, m] > 0 and G[m] > 0 and cred_loja_mes.loc[l, m] <= 0]
        if len(falhas) >= 3:
            alertas.append(dict(tipo="Crédito não recebido", loja=l, kwh=None,
                                detalhe=f"com % > 0, não recebeu crédito em {len(falhas)} dos últimos 12 meses "
                                        f"({', '.join(str(m) for m in falhas[:6])}{'…' if len(falhas) > 6 else ''}) — conferir com a distribuidora"))
    lojas_por_uc = por_uc.groupby("uc")["loja"].first()
    for uc in proprio:
        if uc in lojas_por_uc.index:
            l = lojas_por_uc[uc]
            cob = float(por_uc.loc[por_uc["uc"] == uc, "proprio"].sum())
            alertas.append(dict(tipo="Geração própria", loja=l, kwh=cob,
                                detalhe="a UC também é geradora; só recebe créditos para o consumo que a própria "
                                        "geração não cobre (kWh = geração própria nos meses analisados)"))
    for n in entrada.get("novas") or []:
        alertas.append(dict(tipo="UC nova sem histórico", loja=f"UC {n['uc']}", kwh=None,
                            detalhe=f"{n.get('endereco') or ''} — fora do rateio; definir % por loja análoga"))
    g_df = df[ger]
    saldo_ger = float(g_df[g_df["ref"] == g_df["ref"].max()]["saldo_fin"].sum()) if not g_df.empty else 0.0
    if saldo_ger > 0:
        alertas.append(dict(tipo="Saldo na geradora", loja="GERADORA", kwh=saldo_ger,
                            detalhe="saldo acumulado na própria geradora não é distribuído às beneficiárias"))

    logger.info("Etapa 4 | base: %d loja(s) × %d meses (%s a %s); saldo utilizável %s kWh; preso %s kWh.",
                len(lojas), len(meses), meses[0], ult, _r_fmt(S_util.sum()), _r_fmt(preso.sum()))
    cd_mensal = por_uc.groupby("loja")["cd"].sum() / por_uc.groupby("loja")["ref"].nunique()
    return dict(meses=meses, ultimo=ult, C=C, U=U, S_mes=S_mes, G=G, G_dist=G_dist, pct_mes=pct_mes,
                S_total=S_total, preso=preso, S_util=S_util, pct_atual=pct_atual, ucs=ucs,
                sem_compensacao=sem_comp, alertas=alertas, saldo_geradora=saldo_ger, cd_mensal=cd_mensal,
                origem_cd=origem_cd)

### 4.3 Previsão, simulação mensal, cobertura equalizada e métodos comparados

In [ ]:
def projetar_consumo(U12: np.ndarray):
    """Previsão híbrida: perfil dos últimos 12 meses × fator de tendência amortecido.
    Meses iniciais sem consumo (loja/cadastro recente) são preenchidos com a média observada.
    g = média dos 6 meses mais recentes ÷ média dos 6 anteriores (proxy de crescimento — hipótese)."""
    arr = np.array(U12, dtype=float)
    for i in range(arr.shape[0]):
        obs = arr[i] > 0
        if 0 < obs.sum() < arr.shape[1]:
            primeiro = int(np.argmax(obs))
            if primeiro > 0:
                arr[i, :primeiro] = arr[i, obs].mean()
    g = arr[:, 6:].mean(1) / np.maximum(arr[:, :6].mean(1), 1.0)
    g = np.where(arr[:, :6].mean(1) > 0, g, 1.0)
    g = np.clip(1 + TENDENCIA_AMORTECIMENTO * (g - 1), *TENDENCIA_LIMITES)
    return arr * g[:, None], g


def simular(p, U, G, S0):
    """Aplica o % fixo mês a mês. Devolve (saldo[lojas×T], compensado, faturado, recebido)."""
    n, T = U.shape
    S = np.zeros((n, T)); comp = np.zeros((n, T)); rec = np.zeros((n, T))
    s = np.array(S0, dtype=float).copy()
    for t in range(T):
        rec[:, t] = p * G[t]
        disp = s + rec[:, t]
        comp[:, t] = np.minimum(disp, U[:, t])
        s = disp - comp[:, t]
        S[:, t] = s
    return S, comp, U - comp, rec


def water_filling(Uano, S0, Gano, alvo=0.0, ativos=None):
    """Maximiza a cobertura mínima: aᵢ = max(0, k·Uᵢ + alvoᵢ − Sᵢ), Σ aᵢ = Gano (busca binária em k)."""
    Uano = np.asarray(Uano, float); S0 = np.asarray(S0, float)
    alvo = np.broadcast_to(np.asarray(alvo, float), Uano.shape)
    ativos = np.ones_like(Uano, bool) if ativos is None else np.asarray(ativos, bool)
    if not ativos.any() or Uano[ativos].sum() <= 0:
        a = np.where(ativos, 1.0, 0.0)
        return (a / a.sum() if a.sum() else a), float("inf")
    lo, hi = 0.0, max(20.0, 2 * (Gano + S0[ativos].sum()) / Uano[ativos].sum() + 1)
    for _ in range(200):
        k = (lo + hi) / 2
        a = np.where(ativos, np.maximum(0, k * Uano + alvo - S0), 0.0)
        lo, hi = (k, hi) if a.sum() < Gano else (lo, k)
    return (a / a.sum() if a.sum() else a), k


def buffer_sazonal(G12, U12tot):
    """Colchão (em meses de consumo) que cada mês de início exige para não faltar crédito com % fixo,
    e o pico de saldo acumulado durante o ciclo. Linha i = i-ésimo mês da janela de 12 meses."""
    G12, U12tot = np.asarray(G12, float), np.asarray(U12tot, float)
    if G12.sum() <= 0 or U12tot.sum() <= 0:
        return np.zeros((12, 2))
    w = G12 / G12.sum(); u = U12tot / U12tot.sum()
    linhas = []
    for i in range(12):
        d = np.cumsum(np.roll(w - u, -i)) * 12
        linhas.append((max(0.0, -d.min()), max(0.0, d.max())))
    return np.array(linhas)


def metricas(p, U, G, S0, ativos):
    """Métricas objetivas de um % fixo aplicado por 12 meses (só lojas ativas)."""
    S, comp, fat, _ = simular(p, U, G, S0)
    if not ativos.any():
        ativos = np.ones_like(ativos, bool)
    U, S, comp, fat = U[ativos], S[ativos], comp[ativos], fat[ativos]
    Uano = np.maximum(U.sum(1), 1)
    cob = np.minimum(comp.sum(1) / Uano, 1.0)
    Sf = S[:, -1]
    minimo = max(0.0, U.sum() - np.sum(G) * p[ativos].sum() - np.asarray(S0)[ativos].sum())
    return {
        "Faturado (kWh)": fat.sum(),
        "Faturado acima do mínimo (kWh)": max(0.0, fat.sum() - minimo),
        "Saldo final total (kWh)": Sf.sum(),
        "Saldo ocioso >1 mês (kWh)": np.maximum(0, Sf - U[:, -1]).sum(),
        "Saldo final máx. (meses)": (Sf / (Uano / 12)).max(),
        "Lojas cobertura <90%": int((cob < 0.90).sum()),
        "Cobertura mín.": cob.min(),
        "Dispersão cobertura (dp)": cob.std(),
        "Pico de saldo total (kWh)": S.sum(0).max(),
    }


def otimizar(U, G, S0, p0, ativos, w_disp=20000.0):
    """Método 6: minimiza faturado + saldo final acima de 1 mês + w·var(cobertura) na simulação mensal."""
    if minimize is None or ativos.sum() < 2:
        return None
    idx = np.where(ativos)[0]
    Ua, S0a, Uano = U[idx], np.asarray(S0)[idx], np.maximum(U[idx].sum(1), 1)

    def custo(z):
        q = np.exp(z - z.max()); q /= q.sum()
        S, comp, fat, _ = simular(q, Ua, G, S0a)
        cob = comp.sum(1) / Uano
        return fat.sum() + np.maximum(0, S[:, -1] - Ua[:, -1]).sum() + w_disp * cob.var() * len(q)

    r = minimize(custo, np.log(np.maximum(p0[idx], 1e-6)), method="Powell",
                 options={"maxiter": 20000, "xtol": 1e-4})
    q = np.exp(r.x - r.x.max()); q /= q.sum()
    p = np.zeros_like(p0); p[idx] = q
    return p


def _normalizar(v, ativos):
    v = np.where(ativos, np.maximum(np.asarray(v, float), 0), 0.0)
    return v / v.sum() if v.sum() else v


def calcular_metodos(U, C12, Gano, S0, S_total, pct_atual, ativos, alvo):
    """Percentuais (frações, soma 1) de cada método comparado."""
    Uano = U.sum(1)
    Cm = C12.mean(1)
    P = {"Atual": _normalizar(pct_atual, np.ones_like(ativos))}
    P["Legado 12,5×"] = _normalizar(np.maximum(0, (HORIZONTE_MESES + MESES_BUFFER) * Cm - S_total), ativos)
    P["Proporcional ao consumo"] = _normalizar(Cm, ativos)
    P["Necessidade − saldo"] = _normalizar(np.maximum(0, Uano - S0), ativos)
    P["Cobertura equalizada (W)"], k = water_filling(Uano, S0, Gano, alvo, ativos)
    return P, k

### 4.4 Orquestrador da Etapa 4 — `calcular_rateio_anual()` (inclui backtest)

In [ ]:
def _ciclo(meses_12: pd.PeriodIndex, mes_inicio: int) -> list:
    """Índices (0-11) das colunas dos últimos 12 meses reordenados para começar em `mes_inicio`."""
    pos = {m.month: i for i, m in enumerate(meses_12)}
    return [pos[(mes_inicio - 1 + i) % 12 + 1] for i in range(12)]


def backtest_origem_movel(base: dict, ativos_nomes: list) -> pd.DataFrame | None:
    """Backtest real: para cada origem com 12 meses antes e 12 depois, calcula o % só com o passado,
    mantém fixo e aplica na geração/consumo REAIS. Requer ≥ 24 meses de histórico."""
    meses = base["meses"]
    if len(meses) < 24:
        logger.warning("Etapa 4 | backtest com origem móvel requer ≥ 24 meses (há %d). "
                       "Usando apenas simulação prospectiva + estresse.", len(meses))
        return None
    lojas = list(base["U"].index)
    ativos = np.array([l in ativos_nomes for l in lojas])
    Uall, Call, Sall = base["U"].to_numpy(float), base["C"].to_numpy(float), base["S_mes"].to_numpy(float)
    Gall = base["G"].to_numpy(float)
    pct = base["pct_atual"].to_numpy(float)
    linhas = []
    for o in range(12, len(meses) - 11):
        Up, _ = projetar_consumo(Uall[:, o - 12:o])
        S0 = Sall[:, o - 1]
        P, _ = calcular_metodos(Up, Call[:, o - 12:o], Gall[o - 12:o].sum(), S0, S0, pct, ativos, 0.0)
        for nome, p in P.items():
            linhas.append(dict(origem=str(meses[o]), metodo=nome,
                               **metricas(p, Uall[:, o:o + 12], Gall[o:o + 12], S0, ativos)))
    bt = pd.DataFrame(linhas)
    logger.info("Etapa 4 | backtest: %d origens × %d métodos.", bt["origem"].nunique(), bt["metodo"].nunique())
    return bt


def calcular_rateio_anual(entrada: dict) -> dict:
    """Etapa 4 completa. `entrada` vem do adaptador do notebook (formato padrão)."""
    base = montar_base_rateio(entrada)
    lojas = list(base["U"].index)
    n = len(lojas)
    meses12 = base["meses"][-12:]
    if len(meses12) < 12:
        raise RuntimeError(f"O rateio anual precisa de 12 meses de histórico (há {len(meses12)}).")

    excl = set(EXCLUIR_DO_RATEIO) | (set(base["sem_compensacao"]) if AUTO_EXCLUIR_SEM_COMPENSACAO else set())
    ativos = np.array([l not in excl for l in lojas])

    # ---- previsão (ordem de calendário dos últimos 12 meses) e alinhamento ao ciclo
    U12 = base["U"][meses12].to_numpy(float)
    C12 = base["C"][meses12].to_numpy(float)
    Uproj_cal, g = projetar_consumo(U12)
    G_cal = base["G"][meses12].to_numpy(float) * FATOR_GERACAO
    med = np.median(G_cal[G_cal > 0]) if (G_cal > 0).any() else 0.0
    for i in np.where(G_cal > 2.2 * med)[0] if med else []:
        viz = (G_cal[(i - 1) % 12] + G_cal[(i + 1) % 12]) / 2
        base["alertas"].append(dict(tipo="Geração atípica", loja="—", kwh=float(G_cal[i]),
                                    detalhe=f"{meses12[i]} tem {G_cal[i] / med:.1f}× a mediana mensal — na projeção "
                                            f"foi substituído pela média dos meses vizinhos ({_r_fmt(viz)} kWh)"))
        G_cal[i] = viz
    ordem = _ciclo(meses12, MES_INICIO_CICLO)
    Uc, Gc = Uproj_cal[:, ordem], G_cal[ordem]
    Gano = Gc.sum()

    pct_atual = base["pct_atual"].to_numpy(float)
    total_pct = pct_atual.sum()
    p_atual = pct_atual / total_pct if total_pct else np.full(n, 1 / n)
    G_benef = Gc * (total_pct / 100.0 if total_pct else 1.0)       # o que efetivamente vai às lojas

    # ---- meses entre o último dado e o início do ciclo: simulados com o % atual
    gap = (MES_INICIO_CICLO - (base["ultimo"].month + 1)) % 12
    S0 = base["S_util"].to_numpy(float)
    preso_cont = np.where(ativos, PESO_SALDO_PRESO * base["preso"].to_numpy(float), 0.0)
    if preso_cont.sum() > 0:                       # parte do saldo preso que se espera recuperar
        S0 = S0 + preso_cont
        logger.info("Etapa 4 | saldo preso contado a %.0f%% no rateio: %s kWh.",
                    PESO_SALDO_PRESO * 100, _r_fmt(preso_cont.sum()))
    if gap:
        og = _ciclo(meses12, base["ultimo"].month % 12 + 1)[:gap]
        S_gap, *_ = simular(p_atual, Uproj_cal[:, og], G_cal[og] * (total_pct / 100.0 if total_pct else 1.0), S0)
        S0 = S_gap[:, -1]
    logger.info("Etapa 4 | %s: ciclo começa em %s; %d mês(es) até a troca simulados com o %% atual.",
                entrada.get("sistema", ""), _rotulo_mes(MES_INICIO_CICLO), gap)

    # ---- saldo-alvo: colchão sazonal do mês de início + margem
    buf = buffer_sazonal(G_cal, Uproj_cal[ativos].sum(0) if ativos.any() else Uproj_cal.sum(0))
    buf_por_mes = {m.month: buf[i] for i, m in enumerate(meses12)}
    meses_buffer = buf_por_mes[MES_INICIO_CICLO][0] if USAR_BUFFER_SAZONAL else 0.0
    Uano = Uc.sum(1)
    alvo = Uano * (meses_buffer / 12 + MARGEM_SEGURANCA)

    # ---- métodos + otimização
    P, k = calcular_metodos(Uc, C12, G_benef.sum(), S0, base["S_total"].to_numpy(float),
                            pct_atual, ativos, alvo)
    if RODAR_OTIMIZACAO:
        p_opt = otimizar(Uc, G_benef, S0, P["Cobertura equalizada (W)"], ativos)
        if p_opt is not None:
            P["Otimização numérica"] = p_opt

    linhas = []
    for cen, (fg, fu) in CENARIOS.items():
        for nome, p in P.items():
            linhas.append(dict(cenario=cen, metodo=nome, **metricas(p, Uc * fu, G_benef * fg, S0, ativos)))
    comp_df = pd.DataFrame(linhas)

    # ---- % ideal final (W), arredondado conservando Σ% atual
    pW = P["Cobertura equalizada (W)"]
    pct_ideal = _r_arred({l: pW[i] * total_pct for i, l in enumerate(lojas)})
    pI = np.array([pct_ideal[l] for l in lojas]) / (total_pct if total_pct else 1.0)
    S_prop, comp_prop, fat_prop, rec_prop = simular(pI, Uc, G_benef, S0)
    S_atu, comp_atu, _, _ = simular(p_atual, Uc, G_benef, S0)

    # ---- sobra estrutural: geração + saldo acima do consumo (o saldo vai crescer)
    U_at = Uano[ativos].sum() if ativos.any() else 0.0
    sobra = G_benef.sum() + S0[ativos].sum() - U_at - alvo[ativos].sum()
    necessidade = np.where(ativos, np.maximum(0, np.minimum(k, 1.0) * Uano + alvo - S0), 0.0)
    cedivel = max(0.0, 1 - necessidade.sum() / G_benef.sum()) if G_benef.sum() else 0.0
    if sobra > 0.05 * max(U_at, 1):
        base["alertas"].append(dict(
            tipo="Excedente estrutural", loja="—", kwh=float(sobra),
            detalhe=f"geração do ciclo + saldo superam o consumo compensável — o saldo vai crescer. "
                    f"O saldo atual já cobre ~{S0[ativos].sum() / max(U_at / 12, 1):.0f} meses de consumo; "
                    f"≈ {cedivel * 100:.0f}% da energia do ciclo poderia ser rateada para outras UCs"))

    cd_m = base["cd_mensal"].reindex(lojas).fillna(CUSTO_DISP_PADRAO).to_numpy(float)
    obs = []
    for i, l in enumerate(lojas):
        o = []
        if l in excl:
            o.append("fora do rateio (sem compensação)" if l in base["sem_compensacao"] else "excluída (config)")
        if base["preso"][l] > 0:
            o.append(f"{_r_fmt(base['preso'][l])} kWh presos"
                     + (f" ({PESO_SALDO_PRESO:.0%} contado no rateio)" if ativos[i] and PESO_SALDO_PRESO else ""))
        if g[i] >= 1.10:
            o.append(f"consumo crescendo (+{(g[i] - 1) * 100:.0f}%)")
        if g[i] <= 0.92:
            o.append(f"consumo caindo ({(g[i] - 1) * 100:.0f}%)")
        if ativos[i] and Uano[i] <= 0:
            o.append("consumo coberto pela geração própria")
        elif ativos[i] and pW[i] == 0:
            o.append("saldo cobre o ciclo")
        obs.append("; ".join(o))
    tabela = pd.DataFrame({
        "ucs": base["ucs"].reindex(lojas).values,
        "consumo_anual_proj": Uc.sum(1) + cd_m * 12,
        "custo_disp_anual": cd_m * 12,
        "compensavel_proj": Uano,
        "saldo_atual": base["S_total"].values,
        "saldo_preso": base["preso"].values,
        "saldo_inicio": S0,
        "saldo_alvo": np.where(ativos, alvo, 0.0),
        "necessidade": np.where(ativos, np.maximum(0, k * Uano + alvo - S0) if np.isfinite(k) else 0.0, 0.0),
        "pct_atual": pct_atual,
        "pct_ideal": [pct_ideal[l] for l in lojas],
        "kwh_ideal_ano": pI * G_benef.sum(),
        "kwh_atual_ano": p_atual * G_benef.sum(),
        "saldo_min_proj": S_prop.min(1),
        "saldo_final_proj": S_prop[:, -1],
        "cobertura_proj": comp_prop.sum(1) / np.maximum(Uano, 1),
        "cobertura_atual": comp_atu.sum(1) / np.maximum(Uano, 1),
        "tendencia": g,
        "ativo": ativos,
        "observacao": obs,
    }, index=pd.Index(lojas, name="loja"))

    parametros = {
        "Sistema": entrada.get("sistema", ""),
        "Fonte": entrada.get("fonte", ""),
        "Estratégia": "Cobertura anual equalizada (water-filling)",
        "Início do ciclo": _rotulo_mes(MES_INICIO_CICLO),
        "Base histórica": f"{base['meses'][0]} a {base['ultimo']} ({len(base['meses'])} meses)",
        "Energia distribuível no ciclo (kWh)": Gano,
        "Σ % atual das beneficiárias": total_pct,
        "Energia às lojas no ciclo (kWh)": G_benef.sum(),
        "Consumo compensável projetado (kWh)": U_at,
        "Saldo utilizável no início (kWh)": S0[ativos].sum() if ativos.any() else 0.0,
        "Saldo preso (kWh)": base["preso"].sum(),
        "Cobertura comum k": k if np.isfinite(k) else 0.0,
        "Cobertura estrutural (G ÷ U)": G_benef.sum() / max(U_at, 1),
        "Parcela cedível a outras UCs": cedivel,
        "Colchão sazonal (meses)": meses_buffer,
        "Margem de segurança": MARGEM_SEGURANCA,
        "Peso do saldo preso": PESO_SALDO_PRESO,
        "Fator de geração": FATOR_GERACAO,
        "Amortecimento da tendência": TENDENCIA_AMORTECIMENTO,
        "Custo de disponibilidade": ", ".join(sorted({f"{v}" for v in base["origem_cd"].values()})),
    }
    bt = backtest_origem_movel(base, [l for l, a in zip(lojas, ativos) if a])
    logger.info("Etapa 4 | %s: k = %.1f%% | cobertura estrutural %.1f%% | Σ%% ideal = %.2f",
                entrada.get("sistema", ""), parametros["Cobertura comum k"] * 100,
                parametros["Cobertura estrutural (G ÷ U)"] * 100, sum(pct_ideal.values()))
    return dict(tabela=tabela, metodos=P, comparacao=comp_df, k=parametros["Cobertura comum k"],
                cobertura_estrutural=parametros["Cobertura estrutural (G ÷ U)"],
                parametros=parametros, alertas=base["alertas"], backtest=bt,
                meses_ciclo=[_rotulo_mes(MES_INICIO_CICLO + i) for i in range(12)],
                U_ciclo=Uc, G_ciclo=G_benef, ativos=ativos,
                S_prop=S_prop, S_atual=S_atu, rec_prop=rec_prop, fat_prop=fat_prop,
                buffer={m: buf_por_mes[m] for m in range(1, 13) if m in buf_por_mes},
                base=base, portal=entrada.get("portal", "site da distribuidora"))

## Etapa 5 — Saídas: relatório PDF por demonstrativo (e, opcionalmente, abas `Rateio *` no `.xlsx`)

**Excel**: por padrão (`GERAR_ABAS_RATEIO = False`) a planilha **não** recebe abas de rateio — se existirem de
execuções anteriores, são removidas. Com `GERAR_ABAS_RATEIO = True` as abas abaixo são recriadas a cada execução:

| Aba | Conteúdo |
|---|---|
| `Rateio Anual` | tabela final por loja. `k` (célula `B4`) é editável: **Necessidade**, **% recalculado** e **kWh/ano** são fórmulas. A coluna **% Ideal (digitar no site)** é o valor arredondado que soma exatamente o Σ% atual |
| `Rateio Simulação` | saldo projetado mês a mês de cada loja com o % Ideal e com o % Atual |
| `Rateio Métodos` | métricas de cada método × cenário de estresse, % de cada método e, se houver, o backtest |
| `Rateio Premissas` | parâmetros, premissas, hipóteses e alertas (saldo preso, UCs novas, sobra, etc.) |

**PDF** (um por demonstrativo): 1) painel executivo; 2) rateio proposto; 3) gráficos (sazonalidade,
saldo projetado, colchão por mês de início, cobertura por loja); 4) comparação de métodos; 5) premissas e alertas.

### 5.1 Abas `Rateio *` no Excel

In [ ]:
import openpyxl as _oxl
from openpyxl.styles import Alignment as _Al, Border as _Bo, Font as _Fo, PatternFill as _PF, Side as _Si
from openpyxl.utils import get_column_letter as _col

# mesmo padrão visual das planilhas PR/SC: cabeçalho cinza CCCCCC, texto preto em negrito, aba laranja
_R_HEADER, _R_HEADER2 = "000000", "CCCCCC"
_R_COR_ABA = "FF9900"
_R_FILL_H = _PF("solid", fgColor=_R_HEADER2)
_R_FILL_Z = _PF("solid", fgColor="F2F2F2")
_R_FONT_H = _Fo(bold=True, color="000000")
_R_BORDA = _Bo(bottom=_Si(style="thin", color="D9E2E4"))


def _r_cabecalho(ws, linha, titulos, larguras=None):
    for j, t in enumerate(titulos, 1):
        c = ws.cell(linha, j, t)
        c.fill, c.font = _R_FILL_H, _R_FONT_H
        c.alignment = _Al(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[linha].height = 32
    for j, w in enumerate(larguras or [], 1):
        ws.column_dimensions[_col(j)].width = w


def _r_titulo(ws, texto, sub=""):
    ws["A1"] = texto
    ws["A1"].font = _Fo(bold=True, size=13, color=_R_HEADER)
    ws["A1"].fill = _R_FILL_H
    ws.sheet_properties.tabColor = _R_COR_ABA
    if sub:
        ws["A2"] = sub
        ws["A2"].font = _Fo(italic=True, size=9, color="5A6B6B")


TEXTO_PREMISSAS = {
    "Dados observados": [
        "Consumo, créditos recebidos, % e saldo por UC vêm da aba Demonstrativo (extraída do PDF da distribuidora).",
        "O custo de disponibilidade (TR 100 / BI 50 / MO 30 kWh por mês) nunca é compensado: crédito usado = consumo − custo de disponibilidade.",
        "A geração é sazonal (verão > inverno) e o consumo das lojas costuma ser quase plano.",
    ],
    "Premissas": [
        "O % definido vale por 12 meses a partir do mês de início do ciclo e não é alterado no período.",
        "Todas as UCs pagam tarifa equivalente, então o custo total depende do saldo parado, não de qual UC recebe.",
        "Créditos valem 60 meses; saldo parado não se perde de imediato, mas é dinheiro imobilizado e risco de expirar.",
    ],
    "Hipóteses (validar)": [
        "Geração do próximo ciclo = últimos 12 meses × FATOR_GERACAO.",
        "Consumo do próximo ciclo = perfil dos últimos 12 meses × tendência amortecida por loja.",
        "O saldo preso será tratado com a distribuidora; até lá, não conta como saldo utilizável.",
    ],
    "Como usar": [
        "Digite no site da distribuidora o '% Ideal' da página 'Rateio anual proposto' deste relatório (soma = Σ % atual).",
        "UCs com 0% têm saldo suficiente para o ciclo, geração própria ou estão fora do rateio (ver Observação).",
        "Reavalie antes da próxima troca; com ≥ 24 meses de histórico o backtest roda automaticamente.",
    ],
}


def escrever_abas_rateio(rateio: dict, xlsx_path: Path) -> Path:
    """Recria as abas `Rateio *` na planilha informada."""
    xlsx_path = Path(xlsx_path)
    wb = _oxl.load_workbook(_io.BytesIO(xlsx_path.read_bytes()), data_only=False)
    removidas = [n for n in ABAS_RATEIO if n in wb.sheetnames]
    for nome in removidas:
        del wb[nome]
    if not GERAR_ABAS_RATEIO:                      # rateio só no relatório PDF
        if removidas:
            buf = _io.BytesIO(); wb.save(buf); xlsx_path.write_bytes(buf.getvalue())
            logger.info("Etapa 5 | abas de rateio removidas de %s: %s", xlsx_path.name, ", ".join(removidas))
        return xlsx_path
    tab, par = rateio["tabela"], rateio["parametros"]
    hoje = dt.date.today().strftime("%d/%m/%Y")

    # ------------------------------------------------------------ Rateio Anual
    ws = wb.create_sheet(ABAS_RATEIO[0])
    _r_titulo(ws, f"Rateio anual proposto — {par['Sistema']} — ciclo a partir de {par['Início do ciclo']}",
              f"Gerado em {hoje} · base {par['Base histórica']} · % fixo por 12 meses")
    ws["A4"], ws["B4"] = "Cobertura comum k", rateio["k"]
    ws["A5"], ws["B5"] = "Energia às UCs no ciclo (kWh)", par["Energia às lojas no ciclo (kWh)"]
    ws["A6"], ws["B6"] = "Σ % a distribuir", par["Σ % atual das beneficiárias"]
    ws["B4"].number_format, ws["B5"].number_format, ws["B6"].number_format = "0.0%", "#,##0", "0.00"
    ws["B4"].fill = _PF("solid", fgColor="FFF4D6")
    ws["C4"] = "← editável: altere k (ou use Atingir Meta para Σ Necessidade = B5) e veja as fórmulas"
    ws["C4"].font = _Fo(italic=True, size=9, color="5A6B6B")
    titulos = ["Loja / UC", "UC(s)", "Consumo anual proj. (kWh)", "Custo disp. anual (kWh)",
               "Compensável proj. (kWh)", "Saldo atual (kWh)", "Saldo preso (kWh)",
               "Saldo no início do ciclo (kWh)", "Saldo-alvo (kWh)", "Necessidade (kWh)",
               "% Atual", "% Ideal (digitar no site)", "% Ideal recalc. (fórmula)", "Δ p.p.",
               "kWh ideal / ano", "Saldo mínimo proj. (kWh)", "Saldo final proj. (kWh)",
               "Cobertura proj.", "Cobertura c/ % atual", "Tendência consumo", "Observação"]
    L0 = 8
    _r_cabecalho(ws, L0, titulos, [26, 22, 13, 11, 13, 11, 11, 13, 11, 13, 9, 12, 12, 8,
                                   12, 12, 12, 10, 10, 10, 46])
    tab_ord = tab.sort_values("pct_ideal", ascending=False)
    r1 = L0 + 1
    rN = r1 + len(tab_ord) - 1
    for i, (loja, t) in enumerate(tab_ord.iterrows()):
        r = r1 + i
        nec = f"=IF({'TRUE' if t['ativo'] else 'FALSE'},MAX(0,$B$4*E{r}+I{r}-H{r}),0)"
        vals = [loja, t["ucs"], t["consumo_anual_proj"], t["custo_disp_anual"], t["compensavel_proj"],
                t["saldo_atual"], t["saldo_preso"], t["saldo_inicio"], t["saldo_alvo"], nec,
                t["pct_atual"], t["pct_ideal"], f"=IFERROR(J{r}/SUM($J${r1}:$J${rN})*$B$6,0)",
                f"=L{r}-K{r}", f"=IFERROR(L{r}/$B$6*$B$5,0)", t["saldo_min_proj"], t["saldo_final_proj"],
                t["cobertura_proj"], t["cobertura_atual"], t["tendencia"] - 1, t["observacao"]]
        for j, v in enumerate(vals, 1):
            c = ws.cell(r, j, v.item() if hasattr(v, "item") else v)
            c.border = _R_BORDA
            if i % 2:
                c.fill = _R_FILL_Z
            if 3 <= j <= 10 or j in (15, 16, 17):
                c.number_format = "#,##0"
            elif j in (11, 12, 13, 14):
                c.number_format = "0.00"
            elif j in (18, 19):
                c.number_format = "0.0%"
            elif j == 20:
                c.number_format = "+0%;-0%;0%"
        ws.cell(r, 12).font = _Fo(bold=True)
    rt = rN + 1
    ws.cell(rt, 1, "TOTAL").font = _Fo(bold=True)
    for j in (3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15):
        L = _col(j)
        c = ws.cell(rt, j, f"=SUM({L}{r1}:{L}{rN})")
        c.font = _Fo(bold=True)
        c.number_format = "0.00" if j in (11, 12, 13) else "#,##0"
    ws.cell(rt + 1, 1, "Conferência: Σ % Ideal = Σ % a distribuir")
    ws.cell(rt + 1, 12, f'=IF(ROUND(L{rt}-$B$6,2)=0,"OK","DIFERENÇA")')
    ws.freeze_panes = ws.cell(r1, 2)

    # ------------------------------------------------------------ Rateio Simulação
    ws = wb.create_sheet(ABAS_RATEIO[1])
    _r_titulo(ws, "Saldo projetado mês a mês (kWh) — % fixo por 12 meses",
              "Hipótese: geração e consumo do ciclo repetem o perfil dos últimos 12 meses (com tendência).")
    lojas = list(tab.index)
    meses = rateio["meses_ciclo"]
    linha = 4
    for rotulo, S in [("Com % Ideal", rateio["S_prop"]), ("Com % Atual", rateio["S_atual"])]:
        ws.cell(linha, 1, rotulo).font = _Fo(bold=True, color=_R_HEADER)
        linha += 1
        _r_cabecalho(ws, linha, ["Loja / UC", "Início"] + meses, [26, 10] + [9] * 12)
        for i, l in enumerate(lojas):
            r = linha + 1 + i
            ws.cell(r, 1, l)
            ws.cell(r, 2, float(tab.loc[l, "saldo_inicio"])).number_format = "#,##0"
            for t in range(12):
                c = ws.cell(r, 3 + t, float(S[i, t]))
                c.number_format = "#,##0"
                if S[i, t] < 1 and rateio["U_ciclo"][i, t] > 0:
                    c.fill = _PF("solid", fgColor="FDECEC")
        r = linha + 1 + len(lojas)
        ws.cell(r, 1, "TOTAL").font = _Fo(bold=True)
        for j in range(2, 15):
            L = _col(j)
            c = ws.cell(r, j, f"=SUM({L}{linha + 1}:{L}{r - 1})")
            c.font, c.number_format = _Fo(bold=True), "#,##0"
        linha = r + 3
    ws.cell(linha, 1, "Células em vermelho: mês com saldo zerado (parte do consumo vai para a fatura).").font = \
        _Fo(italic=True, size=9, color="7A2E2E")

    # ------------------------------------------------------------ Rateio Métodos
    ws = wb.create_sheet(ABAS_RATEIO[2])
    _r_titulo(ws, "Comparação de métodos — % fixo por 12 meses sob cenários de estresse",
              "Menor é melhor em faturado, saldo ocioso e dispersão; maior é melhor em cobertura mínima.")
    comp = rateio["comparacao"]
    cols = [c for c in comp.columns if c not in ("cenario", "metodo")]
    _r_cabecalho(ws, 4, ["Cenário", "Método"] + cols, [26, 26] + [14] * len(cols))
    for i, (_, rr) in enumerate(comp.iterrows()):
        r = 5 + i
        ws.cell(r, 1, rr["cenario"]); ws.cell(r, 2, rr["metodo"])
        for j, c in enumerate(cols, 3):
            cel = ws.cell(r, j, float(rr[c]))
            cel.number_format = "0.0%" if "Cobertura" in c or "Dispersão" in c else (
                "0.0" if "meses" in c else "#,##0")
            if rr["metodo"] == "Cobertura equalizada (W)":
                cel.font = _Fo(bold=True)
    r = 5 + len(comp) + 2
    ws.cell(r, 1, "% por método (soma = Σ % a distribuir)").font = _Fo(bold=True, color=_R_HEADER)
    metodos = list(rateio["metodos"])
    _r_cabecalho(ws, r + 1, ["Loja / UC"] + metodos)
    tp = par["Σ % atual das beneficiárias"]
    for i, l in enumerate(lojas):
        ws.cell(r + 2 + i, 1, l)
        for j, m in enumerate(metodos, 2):
            ws.cell(r + 2 + i, j, float(rateio["metodos"][m][i] * tp)).number_format = "0.00"
    bt = rateio.get("backtest")
    r = r + 4 + len(lojas)
    ws.cell(r, 1, "Backtest com origem móvel").font = _Fo(bold=True, color=_R_HEADER)
    if bt is None or bt.empty:
        ws.cell(r + 1, 1, "Indisponível: requer ≥ 24 meses de histórico. Rode de novo quando o histórico completar 24 meses.")
    else:
        _r_cabecalho(ws, r + 1, list(bt.columns))
        for i, (_, rr) in enumerate(bt.iterrows()):
            for j, c in enumerate(bt.columns, 1):
                ws.cell(r + 2 + i, j, rr[c] if isinstance(rr[c], str) else float(rr[c]))

    # ------------------------------------------------------------ Rateio Premissas
    ws = wb.create_sheet(ABAS_RATEIO[3])
    _r_titulo(ws, f"Premissas, parâmetros e alertas do rateio anual — {par['Sistema']}")
    _r_cabecalho(ws, 3, ["Parâmetro", "Valor"], [44, 70])
    for i, (kk, v) in enumerate(par.items()):
        ws.cell(4 + i, 1, kk)
        c = ws.cell(4 + i, 2, v.item() if hasattr(v, "item") else v)
        if isinstance(v, float):
            c.number_format = ("0.0%" if kk.startswith(("Cobertura", "Margem", "Parcela"))
                               else ("0.00" if abs(v) < 50 else "#,##0"))
    r = 5 + len(par)
    for bloco, itens in TEXTO_PREMISSAS.items():
        ws.cell(r, 1, bloco).font = _Fo(bold=True, color=_R_HEADER)
        for it in itens:
            r += 1
            ws.cell(r, 1, "• " + it).alignment = _Al(wrap_text=True, vertical="top")
            ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
            ws.row_dimensions[r].height = 30
        r += 2
    ws.cell(r, 1, "Alertas").font = _Fo(bold=True, color="B23B3B")
    _r_cabecalho(ws, r + 1, ["Tipo / Loja", "Detalhe"])
    for i, a in enumerate(rateio["alertas"]):
        ws.cell(r + 2 + i, 1, f"{a['tipo']} — {a['loja']}")
        kwh = f" ({_r_fmt(a['kwh'])} kWh)" if a.get("kwh") else ""
        ws.cell(r + 2 + i, 2, a["detalhe"] + kwh).alignment = _Al(wrap_text=True)

    buf = _io.BytesIO()
    wb.save(buf)
    xlsx_path.write_bytes(buf.getvalue())
    logger.info("Etapa 5 | abas %s gravadas em %s", ", ".join(ABAS_RATEIO), xlsx_path.name)
    return xlsx_path

### 5.2 Relatório PDF (painel + rateio + gráficos + métodos + premissas)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.patches import FancyBboxPatch
from matplotlib.backends.backend_pdf import PdfPages

R_FUNDO, R_HEADER, R_HEADER2, R_TXT = "#f4f6f8", "#0f3d3e", "#155e63", "#1c2b2b"
R_LINHA_A, R_LINHA_B = "#ffffff", "#eef2f3"
COR_PROP, COR_ATUAL = "#155e63", "#c98a1a"      # séries "% Ideal" e "% Atual"
COR_NEUTRA, COR_GRADE = "#9aa7a7", "#d9e2e4"


def _pagina_base(titulo: str, subtitulo: str, altura: float = 11.0):
    fig = plt.figure(figsize=(13.6, altura), dpi=140)
    fig.patch.set_facecolor(R_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    H = altura
    ax.add_patch(FancyBboxPatch((0.03, 1 - 1.0 / H), 0.94, 0.85 / H,
                 boxstyle="round,pad=0.002,rounding_size=0.006", fc=R_HEADER, ec="none", zorder=2))
    ax.text(0.05, 1 - 0.43 / H, titulo, fontsize=17, fontweight="bold", color="white", va="center", zorder=3)
    ax.text(0.05, 1 - 0.78 / H, subtitulo, fontsize=9.5, color="#cfe3e3", va="center", zorder=3)
    return fig, ax, H


def _estilo_eixo_r(a):
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        a.spines[s].set_color(COR_GRADE)
    a.tick_params(colors="#5a6b6b", labelsize=8)
    a.grid(axis="y", color=COR_GRADE, lw=0.6)
    a.set_axisbelow(True)
    a.set_facecolor("white")


def _quebrar(texto: str, largura: int) -> list:
    palavras, linhas, atual = texto.split(), [], ""
    for p in palavras:
        if len(atual) + len(p) + 1 > largura:
            linhas.append(atual); atual = ""
        atual += (" " if atual else "") + p
    return linhas + ([atual] if atual else [])


def pagina_painel_png(png: Path):
    """Página 1 a partir do PNG do painel já gerado pelo notebook."""
    img = plt.imread(str(png))
    h, w = img.shape[:2]
    fig = plt.figure(figsize=(13.6, 13.6 * h / w), dpi=140)
    fig.patch.set_facecolor(R_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.imshow(img)
    return fig


def pagina_tabela_rateio(rateio: dict):
    tab = rateio["tabela"].sort_values("pct_ideal", ascending=False)
    par = rateio["parametros"]
    n = len(tab)
    row_h = 0.26
    H = 3.4 + max(n, 4) * row_h
    fig, ax, H = _pagina_base(
        f"Rateio anual proposto — {par['Sistema']} — ciclo a partir de {par['Início do ciclo']}",
        f"Cobertura comum k = {rateio['k'] * 100:.1f}%  ·  energia às UCs no ciclo ≈ "
        f"{_r_fmt(par['Energia às lojas no ciclo (kWh)'])} kWh  ·  Σ% = {par['Σ % atual das beneficiárias']:.2f}"
        f"  ·  % fixo por 12 meses", H)
    def y(px): return 1 - px / H
    X = dict(comp=0.30, util=0.365, preso=0.43, atual=0.485, ideal=0.545, dk=0.62, smin=0.685,
             sfin=0.75, cob=0.835, obs=0.845)
    cols = [("Loja / UC", 0.045, "left"), ("Compensável\nproj. (kWh/ano)", X["comp"], "right"),
            ("Saldo útil\n(kWh)", X["util"], "right"), ("Saldo preso\n(kWh)", X["preso"], "right"),
            ("% Atual", X["atual"], "right"), ("% Ideal", X["ideal"], "right"), ("Δ kWh\n/ano", X["dk"], "right"),
            ("Saldo mín.\nproj.", X["smin"], "right"), ("Saldo final\nproj.", X["sfin"], "right"),
            ("Cobertura\natual → proj.", X["cob"], "right"), ("Observação", X["obs"], "left")]
    yh = 1.25
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.5)), 0.94, 0.5 / H, fc=R_HEADER2, ec="none", zorder=2))
    for nome, cx, al in cols:
        ax.text(cx, y(yh + 0.25), nome, fontsize=7.8, fontweight="bold", color="white",
                ha=al, va="center", zorder=3, linespacing=0.9)
    y0 = yh + 0.5
    for i, (loja, t) in enumerate(tab.iterrows()):
        yy = y0 + i * row_h
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H,
                     fc=R_LINHA_A if i % 2 == 0 else R_LINHA_B, ec="none", zorder=1))
        yc = y(yy + row_h / 2)
        kw = dict(fontsize=8, va="center", zorder=3, color=R_TXT if t["ativo"] else "#9aa7a7")
        ax.text(0.045, yc, str(loja)[:40], ha="left", **kw)
        ax.text(X["comp"], yc, _r_fmt(t["compensavel_proj"]), ha="right", **kw)
        ax.text(X["util"], yc, _r_fmt(t["saldo_inicio"]), ha="right", **kw)
        ax.text(X["preso"], yc, _r_fmt(t["saldo_preso"]) if t["saldo_preso"] else "—", ha="right",
                **{**kw, "color": "#b23b3b" if t["saldo_preso"] else "#9aa7a7"})
        ax.text(X["atual"], yc, f"{t['pct_atual']:.2f}", ha="right", **kw)
        d = t["pct_ideal"] - t["pct_atual"]
        ax.text(X["ideal"], yc, f"{t['pct_ideal']:.2f}", ha="right", fontweight="bold", **kw)
        seta, cs = ("▲", "#1a7f5a") if d > 0.05 else (("▼", "#b23b3b") if d < -0.05 else ("=", "#5a6b6b"))
        ax.text(X["ideal"] + 0.005, yc, seta, fontsize=6.4, ha="left", va="center", color=cs, zorder=3)
        dk = t["kwh_ideal_ano"] - t["kwh_atual_ano"]
        ax.text(X["dk"], yc, _r_fmt_sinal(dk), ha="right",
                **{**kw, "color": "#1a7f5a" if dk > 0 else ("#b23b3b" if dk < 0 else "#5a6b6b")})
        ax.text(X["smin"], yc, _r_fmt(t["saldo_min_proj"]), ha="right", **kw)
        ax.text(X["sfin"], yc, _r_fmt(t["saldo_final_proj"]), ha="right", **kw)
        cob_a, cob_p = min(t["cobertura_atual"], 9.99), min(t["cobertura_proj"], 9.99)
        ax.text(X["cob"], yc, f"{cob_a * 100:.0f}% → {cob_p * 100:.0f}%", ha="right", **kw)
        ax.text(X["obs"], yc, (t["observacao"][:30] + "…") if len(t["observacao"]) > 31 else t["observacao"],
                ha="left", fontsize=6.2, va="center", color="#7a2e2e", zorder=3)
    fy = y0 + max(n, 4) * row_h + 0.3
    ax.text(0.03, y(fy),
            "Compensável = consumo − custo de disponibilidade − geração própria.  Saldo útil = saldo no início do ciclo "
            "sem o saldo preso.  Em cinza: fora do rateio.  Δ kWh/ano = energia recebida no ciclo com % Ideal − com % Atual.",
            fontsize=7, color="#8a9797", va="center")
    return fig


def pagina_graficos(rateio: dict):
    fig, ax0, H = _pagina_base("Por que o % precisa aguentar o ano inteiro",
                               f"{rateio['parametros']['Sistema']} · sazonalidade da geração, trajetória do saldo com % fixo, "
                               "colchão por mês de início e cobertura anual por UC", 11.5)
    gs = fig.add_gridspec(3, 2, left=0.07, right=0.97, top=1 - 1.35 / H, bottom=0.06,
                          hspace=0.55, wspace=0.28, width_ratios=[1.25, 1])
    meses = rateio["meses_ciclo"]; x = np.arange(12)
    at = rateio["ativos"] if rateio["ativos"].any() else np.ones_like(rateio["ativos"])
    a = fig.add_subplot(gs[0, 0]); _estilo_eixo_r(a)
    G = rateio["G_ciclo"]; U = rateio["U_ciclo"][at].sum(0)
    a.plot(x, G / max(G.sum(), 1) * 100, color=COR_PROP, lw=2, marker="o", ms=4, label="Energia distribuída")
    if U.sum() > 0:
        a.plot(x, U / U.sum() * 100, color=COR_ATUAL, lw=2, marker="o", ms=4, label="Consumo compensável")
    a.axhline(100 / 12, color=COR_NEUTRA, lw=0.8, ls="--")
    a.set_xticks(x, meses); a.set_ylabel("% do total anual", fontsize=8, color="#5a6b6b")
    a.set_title("Geração × consumo ao longo do ciclo", fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    a.legend(frameon=False, fontsize=7.5, loc="upper right")
    b = fig.add_subplot(gs[1, 0]); _estilo_eixo_r(b)
    b.plot(x, rateio["S_atual"][at].sum(0), color=COR_ATUAL, lw=2, marker="o", ms=4, label="% Atual")
    b.plot(x, rateio["S_prop"][at].sum(0), color=COR_PROP, lw=2, marker="o", ms=4, label="% Ideal")
    b.set_xticks(x, meses); b.set_ylabel("kWh", fontsize=8, color="#5a6b6b")
    b.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: _r_fmt(v)))
    b.set_title("Saldo total projetado (% fixo)", fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    b.legend(frameon=False, fontsize=7.5)
    c = fig.add_subplot(gs[2, 0]); _estilo_eixo_r(c)
    buf = rateio["buffer"]
    vals = [buf.get(m, (0, 0))[0] for m in range(1, 13)]
    cores = [COR_PROP if m == MES_INICIO_CICLO else "#b9c6c6" for m in range(1, 13)]
    c.bar(np.arange(12), vals, color=cores, width=0.7, edgecolor=R_FUNDO, lw=1)
    c.set_xticks(np.arange(12), [_rotulo_mes(m) for m in range(1, 13)])
    c.set_ylabel("meses de consumo", fontsize=8, color="#5a6b6b")
    c.set_title(f"Colchão de saldo exigido por mês de troca (atual: {_rotulo_mes(MES_INICIO_CICLO)})",
                fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    for i, v in enumerate(vals):
        c.text(i, v + 0.03, f"{v:.1f}", ha="center", fontsize=7, color="#5a6b6b")
    i_esc = MES_INICIO_CICLO - 1
    c.annotate("mês da troca", (i_esc, vals[i_esc]), xytext=(i_esc, max(max(vals), 0.2) * 0.55), ha="center",
               fontsize=7.5, color=COR_PROP, fontweight="bold",
               arrowprops=dict(arrowstyle="-|>", color=COR_PROP, lw=1))
    d = fig.add_subplot(gs[:, 1]); _estilo_eixo_r(d); d.grid(axis="y", visible=False)
    d.grid(axis="x", color=COR_GRADE, lw=0.6)
    tab = rateio["tabela"][rateio["tabela"]["ativo"] & (rateio["tabela"]["compensavel_proj"] > 0)]
    tab = tab.sort_values("cobertura_atual", ascending=True)       # pior cobertura atual no topo
    n_lin = max(len(tab), 8)
    yy = n_lin - 1 - np.arange(len(tab))
    ca = np.minimum(tab["cobertura_atual"].to_numpy() * 100, 100)
    cp = np.minimum(tab["cobertura_proj"].to_numpy() * 100, 100)
    d.hlines(yy, np.minimum(ca, cp), np.maximum(ca, cp), color=COR_GRADE, lw=2)
    d.scatter(ca, yy, color=COR_ATUAL, s=26, zorder=3, label="% Atual", edgecolor="white", lw=0.8)
    d.scatter(cp, yy, color=COR_PROP, s=26, zorder=4, label="% Ideal", edgecolor="white", lw=0.8)
    d.set_yticks(yy, [str(i)[:30] for i in tab.index], fontsize=6.8 if len(tab) > 12 else 8)
    d.set_ylim(-0.6, n_lin - 0.4)
    d.set_xlabel("cobertura do consumo compensável no ciclo (%)", fontsize=8, color="#5a6b6b")
    d.set_title("Cobertura anual por UC", fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    d.legend(frameon=False, fontsize=7.5, loc="lower right")
    return fig


def pagina_metodos(rateio: dict):
    comp = rateio["comparacao"]
    base = comp[comp["cenario"] == "Base"].set_index("metodo")
    pior = comp.groupby("metodo").agg({"Faturado acima do mínimo (kWh)": "max",
                                       "Saldo ocioso >1 mês (kWh)": "max", "Cobertura mín.": "min"})
    metodos = list(base.index)
    fig, ax, H = _pagina_base(f"Comparação de métodos — {rateio['parametros']['Sistema']} — % fixo por 12 meses",
                              "Cenário base (perfil dos últimos 12 meses) e pior caso entre os cenários de estresse: "
                              + ", ".join(CENARIOS), 8.5)
    def y(px): return 1 - px / H
    cols = [("Método", 0.045, "left"), ("Faturado\n(kWh)", 0.31, "right"),
            ("Faturado acima\ndo mínimo", 0.40, "right"), ("Saldo final\n(kWh)", 0.48, "right"),
            ("Saldo ocioso\n>1 mês", 0.56, "right"), ("Cobertura\nmínima", 0.63, "right"),
            ("Dispersão\ncobertura", 0.70, "right"),
            ("PIOR CASO\nfat. acima mín.", 0.80, "right"), ("PIOR CASO\nsaldo ocioso", 0.88, "right"),
            ("PIOR CASO\ncob. mínima", 0.955, "right")]
    yh = 1.3
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.55)), 0.94, 0.55 / H, fc=R_HEADER2, ec="none", zorder=2))
    for nome, cx, al in cols:
        ax.text(cx, y(yh + 0.27), nome, fontsize=7.8, fontweight="bold", color="white", ha=al, va="center",
                zorder=3, linespacing=0.9)
    row_h = 0.42
    for i, m in enumerate(metodos):
        yy = yh + 0.55 + i * row_h
        destaque = m == "Cobertura equalizada (W)"
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H,
                     fc="#e9f6ef" if destaque else (R_LINHA_A if i % 2 == 0 else R_LINHA_B), ec="none", zorder=1))
        yc = y(yy + row_h / 2); bb = base.loc[m]; p = pior.loc[m]
        kw = dict(fontsize=8.6, va="center", zorder=3, color=R_TXT, fontweight="bold" if destaque else "normal")
        vals = [m + ("  ← recomendado" if destaque else ""), _r_fmt(bb["Faturado (kWh)"]),
                _r_fmt(bb["Faturado acima do mínimo (kWh)"]), _r_fmt(bb["Saldo final total (kWh)"]),
                _r_fmt(bb["Saldo ocioso >1 mês (kWh)"]), f"{bb['Cobertura mín.'] * 100:.0f}%",
                f"{bb['Dispersão cobertura (dp)'] * 100:.1f} p.p.",
                _r_fmt(p["Faturado acima do mínimo (kWh)"]), _r_fmt(p["Saldo ocioso >1 mês (kWh)"]),
                f"{p['Cobertura mín.'] * 100:.0f}%"]
        for (nome, cx, al), v in zip(cols, vals):
            ax.text(cx, yc, v, ha=al, **kw)
    fy = yh + 0.55 + len(metodos) * row_h + 0.45
    notas = [
        "Faturado acima do mínimo = energia paga na fatura além do déficit inevitável (consumo − geração − saldo). É o que um rateio melhor pode evitar.",
        "Saldo ocioso = saldo final acima de 1 mês de consumo: crédito parado numa UC enquanto outras pagam fatura.",
        "Cobertura mínima / dispersão = justiça entre UCs. Numa mesma empresa o custo total depende do saldo parado, não de qual UC recebe.",
        "Com poucas UCs ou geração sobrando, os métodos ficam próximos; a diferença aparece no saldo ocioso.",
    ]
    bt = rateio.get("backtest")
    notas.append("Backtest com origem móvel: " + ("indisponível (requer ≥ 24 meses de histórico)."
                                                 if bt is None or bt.empty else
                                                 f"{bt['origem'].nunique()} origens avaliadas — ver aba 'Rateio Métodos'."))
    for i, t in enumerate(notas):
        ax.text(0.04, y(fy + i * 0.32), "• " + t, fontsize=8.2, color="#3c4b4b", va="center")
    return fig


def pagina_premissas(rateio: dict):
    fig, ax, H = _pagina_base(f"Premissas, parâmetros e alertas — {rateio['parametros']['Sistema']}",
                              f"Separação entre dado observado, premissa e hipótese — revise antes de aplicar no {rateio['portal']}",
                              11.0)
    def y(px): return 1 - px / H
    yy = 1.35
    ax.text(0.04, y(yy), "Parâmetros do cálculo", fontsize=11, fontweight="bold", color=R_HEADER2)
    yy += 0.3
    for kk, v in rateio["parametros"].items():
        if isinstance(v, float):
            v = (f"{v * 100:.1f}%" if kk.startswith(("Cobertura", "Margem", "Parcela"))
                 else (f"{v:.2f}" if abs(v) < 50 else f"{_r_fmt(v)}"))
        ax.text(0.05, y(yy), kk, fontsize=8.2, color="#5a6b6b", va="center")
        ax.text(0.28, y(yy), str(v)[:48], fontsize=8.2, color=R_TXT, va="center", fontweight="bold")
        yy += 0.21
    yy2 = 1.35
    ax.text(0.52, y(yy2), "Alertas", fontsize=11, fontweight="bold", color="#b23b3b")
    yy2 += 0.3
    if not rateio["alertas"]:
        ax.text(0.53, y(yy2), "Nenhum alerta.", fontsize=8, color="#5a6b6b", va="center")
    for a in rateio["alertas"][:22]:
        kwh = f" ({_r_fmt(a['kwh'])} kWh)" if a.get("kwh") else ""
        txt = f"{a['tipo']} — {a['loja']}{kwh}: {a['detalhe']}"
        for j, linha in enumerate(_quebrar(txt, 92)):
            ax.text(0.53 if j == 0 else 0.545, y(yy2), ("⚠ " if j == 0 else "") + linha,
                    fontsize=7.6, color="#7a2e2e", va="center")
            yy2 += 0.19
        yy2 += 0.05
    yy = max(yy, yy2) + 0.3
    for bloco, itens in TEXTO_PREMISSAS.items():
        ax.text(0.04, y(yy), bloco, fontsize=10, fontweight="bold", color=R_HEADER2)
        yy += 0.27
        for it in itens:
            for j, linha in enumerate(_quebrar(it, 170)):
                ax.text(0.05, y(yy), ("• " if j == 0 else "   ") + linha, fontsize=8, color="#3c4b4b", va="center")
                yy += 0.2
        yy += 0.12
    ax.text(0.03, y(H - 0.25), f"Gerado automaticamente em {dt.date.today().strftime('%d/%m/%Y')}  ·  "
            f"Fonte: {rateio['parametros']['Fonte']}", fontsize=7, color="#8a9797", va="center")
    return fig


def gerar_relatorio_pdf(rateio: dict, painel=None, saida: Path | None = None) -> Path:
    """Relatório completo em um único PDF: painel + páginas de rateio de cada geradora.
    `rateio`: um rateio ou uma lista deles (um por geradora).
    `painel`: função que devolve a figura do painel, caminho do PNG do painel, ou None."""
    rateios = list(rateio) if isinstance(rateio, (list, tuple)) else [rateio]
    paginas = []
    if callable(painel):
        paginas.append(painel)
    elif painel is not None and Path(painel).exists():
        paginas.append(lambda: pagina_painel_png(Path(painel)))
    for r in rateios:
        paginas += [lambda r=r: pagina_tabela_rateio(r), lambda r=r: pagina_graficos(r),
                    lambda r=r: pagina_metodos(r), lambda r=r: pagina_premissas(r)]
    with PdfPages(str(saida)) as pdf:
        for fazer in paginas:
            fig = fazer()
            pdf.savefig(fig, facecolor=R_FUNDO)
            plt.close(fig)
        info = pdf.infodict()
        info["Title"] = f"Energia Injetada e Rateio Anual — {NOME_RELATORIO}"
    logger.info("Etapa 5 | relatório PDF salvo em: %s", Path(saida).name)
    return Path(saida)

## 6. Execução

Lê **todos** os demonstrativos de `DEMONSTRATIVOS` como um só e gera **uma** planilha; depois roda o rateio
anual de cada geradora, o painel único e o **relatório PDF** único (painel + rateio de cada geradora).


In [ ]:
RESULTADOS = {}
nome = NOME_RELATORIO
logger.info("=" * 70)
logger.info("Relatório: %s (%s)", nome, ", ".join(Path(p).name for p in DEMONSTRATIVOS))
try:
    df_pdf, exp = extrair_pdfs(DEMONSTRATIVOS)
    divergencias = validar(df_pdf)
    xlsx = xlsx_saida(nome)
    base = montar_base(df_pdf, ler_base_existente(xlsx))
    gravar_planilha(base, exp, xlsx)
    sem_nome = sorted(set(base.loc[~base["uc_conces"].isin(NOMES_UC), "uc_conces"]))
    if sem_nome:
        logger.warning("UC(s) fora da aba %r: %s", ABA_CADASTRO, ", ".join(f"{u:015d}" for u in sem_nome))
    RESULTADOS[nome] = dict(base=base, expiracao=exp, divergencias=divergencias, xlsx=xlsx)
except Exception as e:                                        # noqa: BLE001
    logger.exception("Falha ao processar %s: %s", nome, e)

for nome, res in RESULTADOS.items():
    base = res["base"]
    # UCs que também são geradoras: a geração própria abate o consumo no rateio das outras geradoras
    PROPRIA = {}
    for uc, gg in base[base["tipo"] == "Geradora"].groupby("uc_conces"):
        PROPRIA[int(uc)] = gg.groupby(pd.to_datetime(gg["referencia"]))["energia_injetada"].sum()

    # um rateio anual por geradora (a 1ª = a que tem beneficiárias, usada no painel)
    res["rateios"] = {}
    for g in geradoras_ordenadas(base):
        nome_g = nome_uc(g)
        logger.info("=" * 70)
        logger.info("Rateio anual: geradora %s", nome_g)
        try:
            res["rateios"][nome_g] = calcular_rateio_anual(
                entrada_rateio_nelson(nome_g, base, PROPRIA, geradora=g))
        except Exception as e:                                # noqa: BLE001
            logger.exception("Falha no rateio anual de %s: %s", nome_g, e)
    rateio = next(iter(res["rateios"].values()), None)
    res["rateio"] = rateio
    try:
        dados, mes_ref = dados_painel_pr(base, rateio)
        png, _ = gerar_painel_pr(dados, mes_ref, painel_saida(nome),
                                 fontes=", ".join(Path(p).name for p in DEMONSTRATIVOS))
        res["png"], res["painel"] = png, pd.DataFrame(dados)
    except Exception as e:                                    # noqa: BLE001
        logger.exception("Falha ao gerar o painel de %s: %s", nome, e)
    if rateio is not None:
        try:
            escrever_abas_rateio(rateio, res["xlsx"])
        except Exception as e:                                # noqa: BLE001
            logger.exception("Falha ao gravar abas do rateio de %s: %s", nome, e)
        try:
            res["relatorio"] = gerar_relatorio_pdf(list(res["rateios"].values()), res.get("png"),
                                                   relatorio_saida(nome))
        except Exception as e:                                # noqa: BLE001
            logger.exception("Falha ao gerar o relatório de %s: %s", nome, e)

for nome, res in RESULTADOS.items():
    print(f"\n### {nome}  ->  {res['xlsx']}  |  relatório: {res.get('relatorio')}")
    if "painel" in res:
        display(res["painel"][["loja", "consumo", "injetada", "saldo", "pct_atual", "pct_ideal", "status"]])
    if not res["divergencias"].empty:
        display(res["divergencias"])
    for nome_g, r in res.get("rateios", {}).items():
        print(f"\n--- Rateio anual: geradora {nome_g}")
        display(r["tabela"][["compensavel_proj", "saldo_inicio", "pct_atual", "pct_ideal",
                             "saldo_final_proj", "cobertura_proj", "observacao"]])


### Créditos a expirar (conforme o demonstrativo mais recente)

In [ ]:
for nome, res in RESULTADOS.items():
    print(f"\n### {nome}")
    display(res["expiracao"][["unidade", "tipo", "geradora_uc_conces", "expirados_kwh",
                              "prox_expirar_kwh", "prox_expirar_data"]])

### Pré-visualização dos painéis

In [ ]:
from IPython.display import Image
for nome, res in RESULTADOS.items():
    if res.get("png") and Path(res["png"]).exists():
        print(f"\n### {nome}")
        display(Image(filename=str(res["png"])))